# Notebook Overview — Run Representation-Based VideoQA

## Purpose

This notebook performs representation-based Video Question Answering (VideoQA) using precomputed video and text representations. Rather than performing direct multimodal inference, the notebook combines shared `clip_text` representations with either `clip_video` or `autoencoder_video` representations and trains a Fusion MLP classifier to predict the correct multiple-choice answer.

The notebook prepares the evaluation dataset, loads the required representation artifacts, constructs multimodal training examples, trains the shared Fusion MLP classifier, generates multiple-choice predictions, validates the resulting prediction dataset, and saves experiment artifacts for downstream evaluation.

The prediction pipeline is identical for both representation-based experiments. The only experimental difference is the source of the video representations.

## Inputs

* Shared `clip_text` representation artifacts
* `clip_video` or `autoencoder_video` representation artifacts
* NExT-QA evaluation annotations
* Shared project configuration

## Outputs

* Multiple-choice prediction dataset
* Prediction validation report
* Experiment summary report
* Saved prediction artifacts for Notebook 08

## Processing Workflow

1. Initialize the notebook environment.
2. Configure the representation-based VideoQA experiment.
3. Verify runtime readiness.
4. Prepare the NExT-QA evaluation dataset.
5. Load shared `clip_text` representations.
6. Load the selected video representations.
7. Prepare multimodal training examples.
8. Train the Fusion MLP classifier and generate multiple-choice predictions.
9. Validate prediction artifacts.
10. Save experiment artifacts.
11. Generate prediction summary reports.
12. Display representative predictions.
13. Summarize the completed experiment.

## Downstream Consumer

Notebook 08 — Evaluate Development Results


### 🔷 Step 1 — Initialize Environment for Representation-Based VideoQA

* Mount Google Drive and prepare the Colab execution environment.
* Clone the private project repository and move into the repository directory.
* Load shared project configuration constants and utility modules.
* Verify required project paths and initialize local output directories.
* Load NExT-QA annotation metadata for evaluation dataset preparation.
* Verify that the required shared CLIP text and video representation artifacts are available.
* Display dataset split summary information when verbose output is enabled.
* Prepare the notebook for representation-based VideoQA inference.


In [ ]:
# ============================================================
# Step 1: Initialize Environment for Representation-Based VideoQA
# ============================================================

VERBOSE = True

import os
from pathlib import Path
import shutil

import pandas as pd
from google.colab import drive, userdata

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# REPO SETUP
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):
    print("Cloning project repository...")
    !git clone --quiet --filter=blob:none --no-checkout {repo_url}
    os.chdir(REPO_DIR)
    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main
else:
    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD CONFIG
# ------------------------------------------------------------

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# EXPERIMENT SETUP
# ------------------------------------------------------------

# EXPERIMENT_NAME = "ae_seg6s_stride4_dev100"
# EXPERIMENT_NAME = "clip_cosine_dev100"
# EXPERIMENT_NAME = "clip_mlp_dev100"
# EXPERIMENT_NAME = "clip_interaction_dev100"
# EXPERIMENT_NAME = "clip_gated_dev100"
EXPERIMENT_NAME = "clip_bilinear_dev100"

configure_experiment(EXPERIMENT_NAME)

EXPERIMENT_TYPE = infer_experiment_type(EXPERIMENT_NAME)

print(f"Experiment name : {EXPERIMENT_NAME}")
print(f"Experiment type : {EXPERIMENT_TYPE}")

# ------------------------------------------------------------
# REQUIRED PATHS
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    Path("src"),
    Path("datasets"),
    QUESTIONS_DIR,
    METADATA_DIR,
    OUTPUTS_DIR,
]

missing_paths = [p for p in required_paths if not p.exists()]

if missing_paths:
    raise FileNotFoundError(f"Missing paths: {missing_paths}")

# ------------------------------------------------------------
# LOAD NExT-QA
# ------------------------------------------------------------

print("\nLoading NExT-QA annotations...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(split_annotations, verbose=VERBOSE)

split_summary_df = summarize_nextqa_splits(annotations_df)

if annotations_df.empty:
    raise RuntimeError("No NExT-QA annotation records loaded.")

# ------------------------------------------------------------
# ARTIFACT RESTORE (STRICT RULE ENFORCEMENT)
# ------------------------------------------------------------

print("\nRestoring representation artifacts...")

# ------------------------------------------------------------
# ALWAYS COPY: CLIP TEXT
# ------------------------------------------------------------

artifact_restore_pairs = [
    {
        "name": "CLIP text representations",
        "drive_path": CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV,
        "local_path": CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
    }
]

# ------------------------------------------------------------
# CONDITIONAL VIDEO SOURCE (ONLY ONE)
# ------------------------------------------------------------

if EXPERIMENT_NAME.startswith("clip"):

    video_name = "CLIP video representations"
    video_drive = CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV
    video_local = CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV

elif EXPERIMENT_NAME.startswith("ae_"):

    video_name = "Autoencoder video representations"
    video_drive = AUTOENCODER_VIDEO_REPRESENTATIONS_CSV
    video_local = AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV

else:
    raise ValueError(
        f"Unknown experiment type for video selection: {EXPERIMENT_NAME}"
    )

artifact_restore_pairs.append({
    "name": video_name,
    "drive_path": video_drive,
    "local_path": video_local,
})

# ------------------------------------------------------------
# COPY ARTIFACTS
# ------------------------------------------------------------

for artifact in artifact_restore_pairs:

    drive_path = Path(artifact["drive_path"])
    local_path = Path(artifact["local_path"])

    if not drive_path.exists():
        raise FileNotFoundError(
            f"Missing Drive artifact: {artifact['name']} -> {drive_path}"
        )

    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        print(f"Copying {artifact['name']} to local runtime...")
        shutil.copy2(drive_path, local_path)
    else:
        print(f"Already exists locally: {local_path}")

print("\nArtifact restoration complete.")

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\nDataset loaded.")
print(f"Annotation records: {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment ready for VideoQA.")



### 🔷 Step 2 — Define Representation-Based VideoQA Configuration

* Configure development-subset or full-dataset execution for representation-based VideoQA.
* Select the active video representation source (`clip_video` or `autoencoder_video`).
* Specify the shared text representation source (`clip_text`).
* Configure the Fusion MLP classifier used for multiple-choice prediction.
* Define the evaluation split, random seed, answer mode, and experiment parameters.
* Configure representation input artifacts and prediction output locations.
* Display the active experiment configuration.


In [ ]:
# ============================================================
# Step 2: Define Representation-Based VideoQA Configuration
# ============================================================

from pathlib import Path

import pandas as pd

print("Defining representation-based VideoQA configuration...")

# ------------------------------------------------------------
# Execution controls
# ------------------------------------------------------------

# Select exactly one supported method from the shared configuration:
#
#   "cosine_similarity"
#   "fusion_mlp_classifier"
#   "interaction_fusion_classifier"
#   "gated_fusion_classifier"
#   "bilinear_fusion_classifier"
#
REPRESENTATION_VIDEOQA_METHOD = "bilinear_fusion_classifier"

RUN_FULL_EVALUATION_SPLIT = False

# ------------------------------------------------------------
# Validate shared method configuration
# ------------------------------------------------------------

required_method_configuration = [
    "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS",
    "REPRESENTATION_VIDEOQA_METHOD_LABELS",
    "REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS",
]

missing_method_configuration = [
    name
    for name in required_method_configuration
    if name not in globals()
]

if missing_method_configuration:
    raise NameError(
        "Missing shared representation-method configuration values: "
        + ", ".join(missing_method_configuration)
    )

if (
    REPRESENTATION_VIDEOQA_METHOD
    not in SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        f"Unsupported REPRESENTATION_VIDEOQA_METHOD: "
        f"{REPRESENTATION_VIDEOQA_METHOD}. "
        f"Expected one of: "
        f"{sorted(SUPPORTED_REPRESENTATION_VIDEOQA_METHODS)}"
    )

representation_videoqa_method_label = (
    REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

representation_videoqa_method_experiment_token = (
    REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

# ------------------------------------------------------------
# Representation source configuration
# ------------------------------------------------------------

TEXT_REPRESENTATION_SOURCE = (
    DEFAULT_TEXT_REPRESENTATION_SOURCE
)

if EXPERIMENT_TYPE == "autoencoder":

    VIDEO_REPRESENTATION_SOURCE = (
        AUTOENCODER_VIDEO_REPRESENTATION_SOURCE
    )

elif EXPERIMENT_TYPE == "clip_video":

    VIDEO_REPRESENTATION_SOURCE = (
        CLIP_VIDEO_REPRESENTATION_SOURCE
    )

else:
    raise ValueError(
        f"Unsupported EXPERIMENT_TYPE: "
        f"{EXPERIMENT_TYPE}"
    )

# ------------------------------------------------------------
# Method/source compatibility validation
# ------------------------------------------------------------

if (
    REPRESENTATION_VIDEOQA_METHOD
    == "cosine_similarity"
):

    if (
        VIDEO_REPRESENTATION_SOURCE
        != CLIP_VIDEO_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity is only supported for "
            "CLIP video representations. "
            f"Current video source: "
            f"{VIDEO_REPRESENTATION_SOURCE}"
        )

    if (
        TEXT_REPRESENTATION_SOURCE
        != DEFAULT_TEXT_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity currently expects "
            "CLIP text representations. "
            f"Current text source: "
            f"{TEXT_REPRESENTATION_SOURCE}"
        )

learned_prediction_methods = {
    "fusion_mlp_classifier",
    "interaction_fusion_classifier",
    "gated_fusion_classifier",
    "bilinear_fusion_classifier",
}

if (
    REPRESENTATION_VIDEOQA_METHOD
    in learned_prediction_methods
):

    if VIDEO_REPRESENTATION_SOURCE not in {
        CLIP_VIDEO_REPRESENTATION_SOURCE,
        AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
    }:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} does not "
            "support the configured video representation "
            f"source: {VIDEO_REPRESENTATION_SOURCE}"
        )

    if (
        TEXT_REPRESENTATION_SOURCE
        != DEFAULT_TEXT_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} currently "
            "expects CLIP text representations. "
            f"Current text source: "
            f"{TEXT_REPRESENTATION_SOURCE}"
        )

# ------------------------------------------------------------
# Development / evaluation configuration
# ------------------------------------------------------------

evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

answer_mode = ANSWER_MODE
choice_columns = CHOICE_COLUMNS
question_column = QUESTION_COLUMN
video_id_column = VIDEO_ID_COLUMN

ground_truth_answer_column = (
    GROUND_TRUTH_ANSWER_COLUMN
)

# ------------------------------------------------------------
# Input artifact configuration
# ------------------------------------------------------------

TEXT_REPRESENTATIONS_CSV = (
    CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV
)

if (
    VIDEO_REPRESENTATION_SOURCE
    == CLIP_VIDEO_REPRESENTATION_SOURCE
):

    VIDEO_REPRESENTATIONS_CSV = (
        CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV
    )

elif (
    VIDEO_REPRESENTATION_SOURCE
    == AUTOENCODER_VIDEO_REPRESENTATION_SOURCE
):

    VIDEO_REPRESENTATIONS_CSV = (
        AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV
    )

else:
    raise ValueError(
        f"Unsupported VIDEO_REPRESENTATION_SOURCE: "
        f"{VIDEO_REPRESENTATION_SOURCE}"
    )

# ------------------------------------------------------------
# Output artifact configuration
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Strict configuration validation
# ------------------------------------------------------------

required_config_values = [
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "TEXT_REPRESENTATION_SOURCE",
    "VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
    "representation_videoqa_method_label",
    "representation_videoqa_method_experiment_token",
    "TEXT_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATIONS_CSV",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "evaluation_split",
    "development_subset_size",
    "random_seed",
    "answer_mode",
    "choice_columns",
    "question_column",
    "video_id_column",
    "ground_truth_answer_column",
]

missing_config_values = [
    name
    for name in required_config_values
    if name not in globals()
]

if missing_config_values:
    raise NameError(
        "Missing required representation-based "
        "VideoQA configuration values: "
        + ", ".join(missing_config_values)
    )

if answer_mode != "multiple_choice":
    raise ValueError(
        f"Unsupported answer mode: {answer_mode}. "
        "Notebook 07 currently supports "
        "multiple_choice only."
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} "
        f"choice columns, found {len(choice_columns)}."
    )

if not Path(
    TEXT_REPRESENTATIONS_CSV
).exists():
    raise FileNotFoundError(
        "Missing text representation artifact: "
        f"{TEXT_REPRESENTATIONS_CSV}"
    )

if not Path(
    VIDEO_REPRESENTATIONS_CSV
).exists():
    raise FileNotFoundError(
        "Missing video representation artifact: "
        f"{VIDEO_REPRESENTATIONS_CSV}"
    )

if not REPRESENTATION_VIDEOQA_LOCAL_DIR.exists():
    raise FileNotFoundError(
        "Representation VideoQA local output "
        "directory was not created: "
        f"{REPRESENTATION_VIDEOQA_LOCAL_DIR}"
    )

# ------------------------------------------------------------
# Temporary experiment-name consistency validation
# ------------------------------------------------------------

expected_method_token = (
    representation_videoqa_method_experiment_token
)

if expected_method_token not in EXPERIMENT_NAME:
    raise ValueError(
        "EXPERIMENT_NAME does not appear to match the "
        "selected prediction method. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}; "
        f"expected experiment token: {expected_method_token}; "
        f"experiment name: {EXPERIMENT_NAME}"
    )

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

representation_videoqa_config_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_type": EXPERIMENT_TYPE,
    "dataset_mode": (
        "full_split"
        if RUN_FULL_EVALUATION_SPLIT
        else "development"
    ),
    "evaluation_split": evaluation_split,
    "development_subset_size": (
        "not applicable"
        if RUN_FULL_EVALUATION_SPLIT
        else development_subset_size
    ),
    "random_seed": random_seed,
    "answer_mode": answer_mode,
    "video_representation_source":
        VIDEO_REPRESENTATION_SOURCE,
    "text_representation_source":
        TEXT_REPRESENTATION_SOURCE,
    "prediction_method":
        REPRESENTATION_VIDEOQA_METHOD,
    "prediction_method_label":
        representation_videoqa_method_label,
    "prediction_method_experiment_token":
        representation_videoqa_method_experiment_token,
    "text_representation_file":
        str(TEXT_REPRESENTATIONS_CSV),
    "video_representation_file":
        str(VIDEO_REPRESENTATIONS_CSV),
    "local_output_directory":
        str(REPRESENTATION_VIDEOQA_LOCAL_DIR),
    "predictions_output_file":
        str(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    "validation_output_file":
        str(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    "summary_output_file":
        str(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
}

representation_videoqa_config_df = pd.DataFrame(
    representation_videoqa_config_summary.items(),
    columns=[
        "Configuration Item",
        "Value",
    ],
)

print(
    "Representation-based VideoQA "
    "configuration defined."
)

display(
    representation_videoqa_config_df
)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify that required runtime objects were initialized by previous steps.
* Confirm that the shared CLIP text and video representation artifacts are available.
* Validate the required NExT-QA annotation schema for representation-based VideoQA.
* Create and verify the local prediction output directory.
* Display runtime environment information and dependency versions.
* Confirm that the notebook is ready to load representation artifacts and perform VideoQA inference.

In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import platform
from pathlib import Path

import numpy as np
import pandas as pd

print("Verifying runtime environment and dependencies...")

# ------------------------------------------------------------
# Verify required in-memory objects
# ------------------------------------------------------------

required_runtime_objects = [
    "annotations_df",
    "split_summary_df",
    "TEXT_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATIONS_CSV",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "TEXT_REPRESENTATION_SOURCE",
    "VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
    "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS",
    "REPRESENTATION_VIDEOQA_METHOD_LABELS",
    "REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS",
    "DEFAULT_TEXT_REPRESENTATION_SOURCE",
    "CLIP_VIDEO_REPRESENTATION_SOURCE",
    "AUTOENCODER_VIDEO_REPRESENTATION_SOURCE",
    "evaluation_split",
    "development_subset_size",
    "answer_mode",
    "choice_columns",
    "question_column",
    "video_id_column",
    "ground_truth_answer_column",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Missing required runtime objects: "
        + ", ".join(missing_runtime_objects)
    )

# ------------------------------------------------------------
# Verify shared prediction-method configuration
# ------------------------------------------------------------

if (
    REPRESENTATION_VIDEOQA_METHOD
    not in SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        "Unsupported REPRESENTATION_VIDEOQA_METHOD. "
        f"Expected one of "
        f"{sorted(SUPPORTED_REPRESENTATION_VIDEOQA_METHODS)}, "
        f"found: {REPRESENTATION_VIDEOQA_METHOD}"
    )

if (
    set(REPRESENTATION_VIDEOQA_METHOD_LABELS)
    != SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        "REPRESENTATION_VIDEOQA_METHOD_LABELS does not match "
        "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS."
    )

if (
    set(REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS)
    != SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
):
    raise ValueError(
        "REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS does not match "
        "SUPPORTED_REPRESENTATION_VIDEOQA_METHODS."
    )

prediction_method_label = (
    REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

prediction_method_experiment_token = (
    REPRESENTATION_VIDEOQA_METHOD_EXPERIMENT_TOKENS[
        REPRESENTATION_VIDEOQA_METHOD
    ]
)

# ------------------------------------------------------------
# Verify representation-source configuration
# ------------------------------------------------------------

if (
    TEXT_REPRESENTATION_SOURCE
    != DEFAULT_TEXT_REPRESENTATION_SOURCE
):
    raise ValueError(
        "Text representation source does not match configuration. "
        f"Found: {TEXT_REPRESENTATION_SOURCE}; "
        f"expected: {DEFAULT_TEXT_REPRESENTATION_SOURCE}"
    )

supported_video_representation_sources = {
    CLIP_VIDEO_REPRESENTATION_SOURCE,
    AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
}

if (
    VIDEO_REPRESENTATION_SOURCE
    not in supported_video_representation_sources
):
    raise ValueError(
        "Unsupported video representation source. "
        f"Expected one of "
        f"{sorted(supported_video_representation_sources)}, "
        f"found: {VIDEO_REPRESENTATION_SOURCE}"
    )

# ------------------------------------------------------------
# Verify method/source compatibility
# ------------------------------------------------------------

if (
    REPRESENTATION_VIDEOQA_METHOD
    == "cosine_similarity"
):

    if (
        VIDEO_REPRESENTATION_SOURCE
        != CLIP_VIDEO_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity requires CLIP video representations. "
            f"Found: {VIDEO_REPRESENTATION_SOURCE}"
        )

    if (
        TEXT_REPRESENTATION_SOURCE
        != DEFAULT_TEXT_REPRESENTATION_SOURCE
    ):
        raise ValueError(
            "cosine_similarity requires the configured CLIP text "
            "representation source."
        )

learned_prediction_methods = (
    SUPPORTED_REPRESENTATION_VIDEOQA_METHODS
    - {"cosine_similarity"}
)

if (
    REPRESENTATION_VIDEOQA_METHOD
    in learned_prediction_methods
):

    if (
        VIDEO_REPRESENTATION_SOURCE
        not in {
            CLIP_VIDEO_REPRESENTATION_SOURCE,
            AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
        }
    ):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} does not support "
            f"video representation source "
            f"{VIDEO_REPRESENTATION_SOURCE}."
        )

# ------------------------------------------------------------
# Verify answer configuration
# ------------------------------------------------------------

if answer_mode != "multiple_choice":
    raise ValueError(
        f"Unsupported answer mode: {answer_mode}. "
        "Notebook 07 supports multiple_choice only."
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer-choice "
        f"columns, found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Verify artifact availability
# ------------------------------------------------------------

required_input_files = [
    Path(TEXT_REPRESENTATIONS_CSV),
    Path(VIDEO_REPRESENTATIONS_CSV),
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_input_files:

    for path in missing_input_files:
        print(f"Missing input artifact: {path}")

    raise FileNotFoundError(
        "One or more required representation artifacts are missing."
    )

# ------------------------------------------------------------
# Verify required annotation columns
# ------------------------------------------------------------

required_annotation_columns = [
    "split",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    *choice_columns,
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        + ", ".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Verify annotation split availability
# ------------------------------------------------------------

available_annotation_splits = set(
    annotations_df["split"]
    .astype(str)
    .unique()
)

required_annotation_splits = {
    "train",
    evaluation_split,
}

missing_annotation_splits = (
    required_annotation_splits
    - available_annotation_splits
)

if missing_annotation_splits:
    raise ValueError(
        "annotations_df is missing required splits: "
        + ", ".join(sorted(missing_annotation_splits))
    )

# ------------------------------------------------------------
# Verify output directory
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not REPRESENTATION_VIDEOQA_LOCAL_DIR.exists():
    raise FileNotFoundError(
        "Output directory was not created: "
        f"{REPRESENTATION_VIDEOQA_LOCAL_DIR}"
    )

# ------------------------------------------------------------
# Verify configured output paths
# ------------------------------------------------------------

configured_output_paths = [
    Path(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    Path(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    Path(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
]

for output_path in configured_output_paths:

    if output_path.parent != Path(
        REPRESENTATION_VIDEOQA_LOCAL_DIR
    ):
        raise ValueError(
            "Configured output artifact is not located inside "
            "REPRESENTATION_VIDEOQA_LOCAL_DIR: "
            f"{output_path}"
        )

# ------------------------------------------------------------
# Display runtime summary
# ------------------------------------------------------------

runtime_summary = {
    "python_version":
        platform.python_version(),

    "pandas_version":
        pd.__version__,

    "numpy_version":
        np.__version__,

    "annotation_records":
        len(annotations_df),

    "available_annotation_splits":
        ", ".join(
            sorted(available_annotation_splits)
        ),

    "text_representation_file":
        str(TEXT_REPRESENTATIONS_CSV),

    "video_representation_file":
        str(VIDEO_REPRESENTATIONS_CSV),

    "text_representation_file_exists":
        Path(TEXT_REPRESENTATIONS_CSV).exists(),

    "video_representation_file_exists":
        Path(VIDEO_REPRESENTATIONS_CSV).exists(),

    "output_directory":
        str(REPRESENTATION_VIDEOQA_LOCAL_DIR),

    "experiment_name":
        EXPERIMENT_NAME,

    "experiment_type":
        EXPERIMENT_TYPE,

    "prediction_method":
        REPRESENTATION_VIDEOQA_METHOD,

    "prediction_method_label":
        prediction_method_label,

    "prediction_method_experiment_token":
        prediction_method_experiment_token,

    "text_representation_source":
        TEXT_REPRESENTATION_SOURCE,

    "video_representation_source":
        VIDEO_REPRESENTATION_SOURCE,
}

runtime_summary_df = pd.DataFrame(
    runtime_summary.items(),
    columns=[
        "Runtime Item",
        "Value",
    ],
)

print("Runtime environment verification complete.")

display(runtime_summary_df)



### 🔷 Step 4 — Prepare NExT-QA Evaluation Dataset

* Select the configured NExT-QA evaluation split.
* Apply development-subset sampling when full-split execution is disabled.
* Reconstruct stable annotation identifiers matching the shared CLIP text representation dataset.
* Normalize video IDs, questions, answer choices, and ground-truth labels.
* Create stable representation-based QA record identifiers.
* Validate required fields, duplicate identifiers, and multiple-choice answer labels.
* Display the prepared QA input dataset for verification.

In [ ]:
# ============================================================
# Step 4: Prepare NExT-QA Evaluation Dataset
# ============================================================

import pandas as pd

print("Preparing NExT-QA evaluation dataset...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "annotations_df" not in globals():
    raise NameError("annotations_df was not found. Run Step 1 first.")

required_annotation_columns = [
    "split",
    "question_id",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    *choice_columns,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        + ", ".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Reconstruct Notebook 05 annotation_id values
# ------------------------------------------------------------

annotations_with_ids_df = annotations_df.copy()

annotations_with_ids_df["annotation_source_index"] = (
    annotations_with_ids_df.index
)

annotations_with_ids_df["annotation_id"] = (
    annotations_with_ids_df["split"].astype(str)
    + "_"
    + annotations_with_ids_df[video_id_column].astype(str)
    + "_"
    + annotations_with_ids_df["question_id"].astype(str)
    + "_"
    + annotations_with_ids_df["annotation_source_index"].astype(str)
)

if "NUM_MULTIPLE_CHOICE_ANSWERS" not in globals():
    raise NameError("NUM_MULTIPLE_CHOICE_ANSWERS was not found in configuration.")

# ------------------------------------------------------------
# Define reusable QA input builder
# ------------------------------------------------------------

def build_qa_input_dataset(split_name):

    split_annotations_df = annotations_with_ids_df[
        annotations_with_ids_df["split"] == split_name
    ].copy()

    if split_annotations_df.empty:
        raise ValueError(
            f"No annotation records found for split: {split_name}"
        )

    if RUN_FULL_EVALUATION_SPLIT:

        split_qa_input_df = (
            split_annotations_df
            .sort_values([video_id_column, question_column])
            .reset_index(drop=True)
        )

    else:

        available_development_video_ids = sorted(
            split_annotations_df[video_id_column]
            .astype(str)
            .unique()
        )

        selected_video_count = min(
            development_subset_size,
            len(available_development_video_ids),
        )

        development_video_ids = (
            pd.Series(available_development_video_ids)
            .sample(
                n=selected_video_count,
                random_state=random_seed,
            )
            .astype(str)
            .tolist()
        )

        if not development_video_ids:
            raise RuntimeError(
                f"No development videos were selected from split: {split_name}"
            )

        split_qa_input_df = (
            split_annotations_df[
                split_annotations_df[video_id_column]
                .astype(str)
                .isin(development_video_ids)
            ]
            .copy()
            .sort_values([video_id_column, question_column])
            .reset_index(drop=True)
        )

    if split_qa_input_df.empty:
        raise RuntimeError(
            f"No representation-based VideoQA input records were generated "
            f"for split: {split_name}"
        )

    # --------------------------------------------------------
    # Normalize key fields
    # --------------------------------------------------------

    split_qa_input_df[video_id_column] = (
        split_qa_input_df[video_id_column]
        .astype(str)
    )

    split_qa_input_df[question_column] = (
        split_qa_input_df[question_column]
        .astype(str)
    )

    split_qa_input_df["annotation_id"] = (
        split_qa_input_df["annotation_id"]
        .astype(str)
    )

    for choice_column in choice_columns:
        split_qa_input_df[choice_column] = (
            split_qa_input_df[choice_column]
            .astype(str)
        )

    split_qa_input_df[ground_truth_answer_column] = (
        split_qa_input_df[ground_truth_answer_column]
        .astype(int)
    )

    # --------------------------------------------------------
    # Add stable QA record id
    # --------------------------------------------------------

    split_qa_input_df = split_qa_input_df.reset_index(drop=True)

    split_qa_input_df["qa_record_id"] = (
        split_qa_input_df["annotation_id"].astype(str)
        + "_representation_videoqa"
    )

    # --------------------------------------------------------
    # Strict validation
    # --------------------------------------------------------

    duplicate_qa_ids = split_qa_input_df["qa_record_id"].duplicated().sum()

    if duplicate_qa_ids > 0:
        raise ValueError(
            f"Found {duplicate_qa_ids} duplicate qa_record_id values "
            f"for split: {split_name}."
        )

    duplicate_annotation_ids = (
        split_qa_input_df["annotation_id"]
        .duplicated()
        .sum()
    )

    if duplicate_annotation_ids > 0:
        raise ValueError(
            f"Found {duplicate_annotation_ids} duplicate annotation_id values "
            f"for split: {split_name}."
        )

    if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
        raise ValueError(
            f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer choices, "
            f"but found {len(choice_columns)}."
        )

    invalid_ground_truth_values = sorted(
        set(split_qa_input_df[ground_truth_answer_column].unique())
        - set(range(NUM_MULTIPLE_CHOICE_ANSWERS))
    )

    if invalid_ground_truth_values:
        raise ValueError(
            "Found invalid ground-truth answer values: "
            + str(invalid_ground_truth_values)
        )

    missing_required_values = (
        split_qa_input_df[
            [
                "qa_record_id",
                "annotation_id",
                "split",
                video_id_column,
                question_column,
                ground_truth_answer_column,
                *choice_columns,
            ]
        ]
        .isna()
        .sum()
        .sum()
    )

    if missing_required_values > 0:
        raise ValueError(
            f"Found {missing_required_values} missing required QA values "
            f"for split: {split_name}."
        )

    return split_annotations_df, split_qa_input_df


# ------------------------------------------------------------
# Build training and validation inputs
# ------------------------------------------------------------

train_annotations_df, train_qa_input_df = build_qa_input_dataset("train")
evaluation_annotations_df, validation_qa_input_df = build_qa_input_dataset(
    evaluation_split
)

# Preserve existing downstream evaluation dataset name
qa_input_df = validation_qa_input_df.copy()

dataset_mode_label = (
    "full_split"
    if RUN_FULL_EVALUATION_SPLIT
    else "development"
)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("NExT-QA training and validation datasets prepared successfully.")
print(f"Dataset mode             : {dataset_mode_label}")
print(f"Training split           : train")
print(f"Validation split         : {evaluation_split}")
print(f"Training source records  : {len(train_annotations_df):,}")
print(f"Validation source records: {len(evaluation_annotations_df):,}")
print(f"Training QA records      : {len(train_qa_input_df):,}")
print(f"Validation QA records    : {len(validation_qa_input_df):,}")
print(f"Training unique videos   : {train_qa_input_df[video_id_column].nunique():,}")
print(f"Validation unique videos : {validation_qa_input_df[video_id_column].nunique():,}")
print(f"Answer mode              : {answer_mode}")
print(f"Answer choices           : {len(choice_columns)}")

print("\nValidation QA Input Preview:")
display(
    validation_qa_input_df[
        [
            "qa_record_id",
            "annotation_id",
            "split",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            *choice_columns,
        ]
    ].head(10)
)



### 🔷 Step 5 — Load CLIP Text Representations

* Load the shared CLIP text representation artifact generated by Notebook 05.
* Validate the required metadata and embedding columns.
* Identify the CLIP text embedding dimensions from the representation dataset.
* Verify that all embedding values are numeric, complete, and internally consistent.
* Confirm that all representation records have unique identifiers and valid input types.
* Filter the shared text representations to the videos and dataset split required for the current evaluation.
* Display a preview of the filtered CLIP text representation dataset for verification.

In [ ]:
# ============================================================
# Step 5: Load CLIP Text Representations
# ============================================================

import pandas as pd
from pathlib import Path

print("Loading CLIP text representations...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step5_objects = [
    "train_qa_input_df",
    "validation_qa_input_df",
    "TEXT_REPRESENTATIONS_CSV",
    "TEXT_REPRESENTATION_SOURCE",
    "TEXT_INPUT_TYPES",
    "CLIP_TEXT_EMBEDDING_DIM",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "choice_columns",
]

missing_step5_objects = [
    name for name in required_step5_objects
    if name not in globals()
]

if missing_step5_objects:
    raise NameError(
        "Missing required Step 5 objects: "
        + ", ".join(missing_step5_objects)
    )

if not Path(TEXT_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing CLIP text representation file: {TEXT_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Load text representation artifact
# ------------------------------------------------------------

clip_text_representation_df = pd.read_csv(TEXT_REPRESENTATIONS_CSV)

if clip_text_representation_df.empty:
    raise RuntimeError(
        "CLIP text representation artifact was loaded but is empty."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_text_columns = [
    "record_id",
    video_id_column,
    "question_id",
    "annotation_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "split",
    "representation_source",
]

missing_text_columns = [
    col for col in required_text_columns
    if col not in clip_text_representation_df.columns
]

if missing_text_columns:
    raise ValueError(
        "CLIP text representation dataset is missing required columns: "
        + ", ".join(missing_text_columns)
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

clip_text_columns = sorted(
    [
        col
        for col in clip_text_representation_df.columns
        if col.startswith("clip_text_")
        and col.replace("clip_text_", "").isdigit()
    ],
    key=lambda col: int(col.replace("clip_text_", ""))
)

text_embedding_dimension = len(clip_text_columns)

if text_embedding_dimension == 0:
    raise ValueError("No CLIP text embedding columns were found.")

if text_embedding_dimension != CLIP_TEXT_EMBEDDING_DIM:
    raise ValueError(
        f"Expected {CLIP_TEXT_EMBEDDING_DIM} CLIP text dimensions, "
        f"found {text_embedding_dimension}."
    )

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

missing_embedding_values = (
    clip_text_representation_df[clip_text_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing CLIP text embedding values."
    )

non_numeric_embedding_columns = [
    col for col in clip_text_columns
    if not pd.api.types.is_numeric_dtype(clip_text_representation_df[col])
]

if non_numeric_embedding_columns:
    raise TypeError(
        "Found non-numeric CLIP text embedding columns: "
        + ", ".join(non_numeric_embedding_columns[:20])
    )

duplicate_text_record_ids = (
    clip_text_representation_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_text_record_ids > 0:
    raise ValueError(
        f"Found {duplicate_text_record_ids} duplicate CLIP text record_id values."
    )

expected_text_types = set(TEXT_INPUT_TYPES)

actual_text_types = set(
    clip_text_representation_df["text_type"]
    .astype(str)
    .unique()
)

unexpected_text_types = actual_text_types - expected_text_types

if unexpected_text_types:
    raise ValueError(
        "Unexpected CLIP text types found: "
        + ", ".join(sorted(unexpected_text_types))
    )

if actual_text_types != {"question_answer"}:
    raise ValueError(
        "Notebook 07 expects CLIP text representations with only "
        "text_type='question_answer'. "
        f"Found: {sorted(actual_text_types)}"
    )

unexpected_sources = set(
    clip_text_representation_df["representation_source"]
    .astype(str)
    .unique()
) - {TEXT_REPRESENTATION_SOURCE}

if unexpected_sources:
    raise ValueError(
        "Unexpected text representation sources found: "
        + ", ".join(sorted(unexpected_sources))
    )

# ------------------------------------------------------------
# Filter to training and validation QA records
# ------------------------------------------------------------

clip_text_representation_df["annotation_id"] = (
    clip_text_representation_df["annotation_id"]
    .astype(str)
)

clip_text_representation_df["split"] = (
    clip_text_representation_df["split"]
    .astype(str)
)

clip_text_representation_df["choice_index"] = (
    clip_text_representation_df["choice_index"]
    .astype(int)
)

clip_text_representation_df["answer"] = (
    clip_text_representation_df["answer"]
    .astype(int)
)

train_qa_input_df["annotation_id"] = (
    train_qa_input_df["annotation_id"]
    .astype(str)
)

validation_qa_input_df["annotation_id"] = (
    validation_qa_input_df["annotation_id"]
    .astype(str)
)

train_annotation_ids = set(train_qa_input_df["annotation_id"])
validation_annotation_ids = set(validation_qa_input_df["annotation_id"])

train_clip_text_df = (
    clip_text_representation_df[
        clip_text_representation_df["annotation_id"].isin(train_annotation_ids)
        &
        (clip_text_representation_df["split"] == "train")
    ]
    .copy()
    .reset_index(drop=True)
)

validation_clip_text_df = (
    clip_text_representation_df[
        clip_text_representation_df["annotation_id"].isin(validation_annotation_ids)
        &
        (clip_text_representation_df["split"] == evaluation_split)
    ]
    .copy()
    .reset_index(drop=True)
)

# Preserve existing downstream evaluation text representation name
filtered_clip_text_df = validation_clip_text_df.copy()

expected_train_text_records = (
    len(train_qa_input_df)
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

expected_validation_text_records = (
    len(validation_qa_input_df)
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

if len(train_clip_text_df) != expected_train_text_records:
    raise ValueError(
        f"Expected {expected_train_text_records:,} training CLIP text "
        f"representation records, found {len(train_clip_text_df):,}."
    )

if len(validation_clip_text_df) != expected_validation_text_records:
    raise ValueError(
        f"Expected {expected_validation_text_records:,} validation CLIP text "
        f"representation records, found {len(validation_clip_text_df):,}."
    )

train_records_per_annotation = (
    train_clip_text_df
    .groupby("annotation_id")
    .size()
)

invalid_train_record_counts = train_records_per_annotation[
    train_records_per_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
]

if len(invalid_train_record_counts) > 0:
    raise ValueError(
        "Each training annotation must have exactly one question-answer "
        "text representation per answer choice."
    )

validation_records_per_annotation = (
    validation_clip_text_df
    .groupby("annotation_id")
    .size()
)

invalid_validation_record_counts = validation_records_per_annotation[
    validation_records_per_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
]

if len(invalid_validation_record_counts) > 0:
    raise ValueError(
        "Each validation annotation must have exactly one question-answer "
        "text representation per answer choice."
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("CLIP question-answer text representations loaded successfully.")
print(f"Source file              : {TEXT_REPRESENTATIONS_CSV}")
print(f"Total text records       : {len(clip_text_representation_df):,}")
print(f"Training text records    : {len(train_clip_text_df):,}")
print(f"Validation text records  : {len(validation_clip_text_df):,}")
print(f"Text types               : {', '.join(sorted(actual_text_types))}")
print(f"Representation source    : {TEXT_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions     : {text_embedding_dimension:,}")
print(f"Answer choices/question  : {NUM_MULTIPLE_CHOICE_ANSWERS}")

print("\nValidation CLIP Text Representation Preview:")
display(
    validation_clip_text_df[
        [
            "record_id",
            video_id_column,
            "question_id",
            "annotation_id",
            "split",
            "text_type",
            "choice_index",
            "text",
            "answer",
            "ground_truth_text",
            "representation_source",
        ]
    ].head(10)
)



### 🔷 Step 6 — Load Video Representations

* Load the configured video representation artifact for representation-based VideoQA.
* Validate required video representation metadata and embedding columns.
* Identify the video embedding dimension from the representation dataset.
* Verify that all video embeddings are numeric, complete, and uniquely keyed by video.
* Filter video representations to the videos required by the current QA input dataset.
* Confirm that every QA video has a matching video representation.
* Display a preview of the filtered video representation dataset for verification.

In [ ]:
# ============================================================
# Step 6: Load Video Representations
# ============================================================

import pandas as pd
from pathlib import Path
import re

print("Loading video representations...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step6_objects = [
    "train_qa_input_df",
    "validation_qa_input_df",
    "VIDEO_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATION_SOURCE",
    "video_id_column",
    "CLIP_VIDEO_EMBEDDING_DIM",
    "AUTOENCODER_VIDEO_EMBEDDING_DIM",
]

missing_step6_objects = [
    name for name in required_step6_objects
    if name not in globals()
]

if missing_step6_objects:
    raise NameError(
        "Missing required Step 6 objects: "
        + ", ".join(missing_step6_objects)
    )

if not Path(VIDEO_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing video representation file: {VIDEO_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Load video representation artifact selected in Step 2
# ------------------------------------------------------------

video_representation_df = pd.read_csv(VIDEO_REPRESENTATIONS_CSV)

if video_representation_df.empty:
    raise RuntimeError(
        "Video representation artifact was loaded but is empty."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_video_representation_columns = [
    "record_id",
    video_id_column,
    "split",
    "representation_source",
]

missing_video_representation_columns = [
    col for col in required_video_representation_columns
    if col not in video_representation_df.columns
]

if missing_video_representation_columns:
    raise ValueError(
        "Video representation dataset is missing required columns: "
        + ", ".join(missing_video_representation_columns)
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

video_embedding_columns = sorted(
    [
        col for col in video_representation_df.columns
        if re.fullmatch(r"embedding_\d{3}", col)
    ],
    key=lambda col: int(col.replace("embedding_", ""))
)

if not video_embedding_columns and VIDEO_REPRESENTATION_SOURCE == CLIP_VIDEO_REPRESENTATION_SOURCE:

    legacy_clip_columns = sorted(
        [
            col for col in video_representation_df.columns
            if re.fullmatch(r"clip_video_\d{3}", col)
        ],
        key=lambda col: int(col.replace("clip_video_", ""))
    )

    if legacy_clip_columns:
        rename_map = {
            old_col: f"embedding_{idx:03d}"
            for idx, old_col in enumerate(legacy_clip_columns)
        }

        video_representation_df = video_representation_df.rename(
            columns=rename_map
        )

        video_embedding_columns = list(rename_map.values())

        print(
            "CLIP video embedding columns standardized from "
            "clip_video_### to embedding_###."
        )

if not video_embedding_columns:
    raise ValueError(
        "No video embedding columns found. Expected standardized "
        "embedding_### columns. For CLIP video artifacts, clip_video_### "
        "columns are also accepted and standardized."
    )

video_embedding_dimension = len(video_embedding_columns)

expected_video_embedding_dimension = (
    CLIP_VIDEO_EMBEDDING_DIM
    if VIDEO_REPRESENTATION_SOURCE == CLIP_VIDEO_REPRESENTATION_SOURCE
    else AUTOENCODER_VIDEO_EMBEDDING_DIM
)

if video_embedding_dimension != expected_video_embedding_dimension:
    raise ValueError(
        f"Expected {expected_video_embedding_dimension} video embedding "
        f"dimensions for {VIDEO_REPRESENTATION_SOURCE}, "
        f"found {video_embedding_dimension}."
    )

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

video_representation_df[video_id_column] = (
    video_representation_df[video_id_column]
    .astype(str)
)

video_representation_df["split"] = (
    video_representation_df["split"]
    .astype(str)
)

missing_embedding_values = (
    video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing video embedding values."
    )

non_numeric_embedding_columns = [
    col for col in video_embedding_columns
    if not pd.api.types.is_numeric_dtype(video_representation_df[col])
]

if non_numeric_embedding_columns:
    raise TypeError(
        "Found non-numeric video embedding columns: "
        + ", ".join(non_numeric_embedding_columns[:20])
    )

duplicate_video_records = (
    video_representation_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_video_records > 0:
    raise ValueError(
        f"Found {duplicate_video_records} duplicate video representation "
        "record_id values."
    )

duplicate_video_split_ids = (
    video_representation_df[
        [
            video_id_column,
            "split",
        ]
    ]
    .duplicated()
    .sum()
)

if duplicate_video_split_ids > 0:
    raise ValueError(
        f"Found {duplicate_video_split_ids} duplicate video/split "
        "representation records."
    )

actual_video_sources = set(
    video_representation_df["representation_source"]
    .astype(str)
    .unique()
)

if actual_video_sources != {VIDEO_REPRESENTATION_SOURCE}:
    raise ValueError(
        f"Expected only video representation source "
        f"{VIDEO_REPRESENTATION_SOURCE}, found: "
        + ", ".join(sorted(actual_video_sources))
    )

# ------------------------------------------------------------
# Filter to training and validation QA videos
# ------------------------------------------------------------

train_qa_input_df[video_id_column] = (
    train_qa_input_df[video_id_column]
    .astype(str)
)

validation_qa_input_df[video_id_column] = (
    validation_qa_input_df[video_id_column]
    .astype(str)
)

train_qa_video_ids = set(
    train_qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

validation_qa_video_ids = set(
    validation_qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

train_video_representation_df = (
    video_representation_df[
        video_representation_df[video_id_column].isin(train_qa_video_ids)
        &
        (video_representation_df["split"] == "train")
    ]
    .copy()
    .reset_index(drop=True)
)

validation_video_representation_df = (
    video_representation_df[
        video_representation_df[video_id_column].isin(validation_qa_video_ids)
        &
        (video_representation_df["split"] == evaluation_split)
    ]
    .copy()
    .reset_index(drop=True)
)

# Preserve existing downstream evaluation video representation name
filtered_video_representation_df = validation_video_representation_df.copy()

if train_video_representation_df.empty:
    raise RuntimeError(
        "No video representations matched the selected training QA records."
    )

if validation_video_representation_df.empty:
    raise RuntimeError(
        "No video representations matched the selected validation QA records."
    )

missing_train_video_ids = sorted(
    train_qa_video_ids
    - set(
        train_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )
)

missing_validation_video_ids = sorted(
    validation_qa_video_ids
    - set(
        validation_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )
)

if missing_train_video_ids:
    raise ValueError(
        "Missing video representations for training QA videos: "
        + ", ".join(missing_train_video_ids[:20])
    )

if missing_validation_video_ids:
    raise ValueError(
        "Missing video representations for validation QA videos: "
        + ", ".join(missing_validation_video_ids[:20])
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("Video representations loaded successfully.")
print(f"Source file                  : {VIDEO_REPRESENTATIONS_CSV}")
print(f"Total video records          : {len(video_representation_df):,}")
print(f"Training video records       : {len(train_video_representation_df):,}")
print(f"Validation video records     : {len(validation_video_representation_df):,}")
print(f"Training QA input videos     : {len(train_qa_video_ids):,}")
print(f"Validation QA input videos   : {len(validation_qa_video_ids):,}")
print(f"Representation source        : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions         : {video_embedding_dimension:,}")

print("\nValidation Video Representation Preview:")
display(
    validation_video_representation_df[
        [
            "record_id",
            video_id_column,
            "split",
            "representation_source",
        ]
    ].head(10)
)



### 🔷 Step 7 — Prepare Representation VideoQA Candidate Datasets

* Load the question, answer-choice, and video representations required for inference.
* Associate each evaluation question with its corresponding video representation.
* Construct one candidate-answer record for each answer choice.
* Prepare multimodal feature vectors for Fusion MLP training and inference.
* Validate dataset completeness before classifier training.


In [ ]:
# ============================================================
# Step 7: Prepare Representation VideoQA Candidate Datasets
# ============================================================

import pandas as pd

print("Preparing representation-based VideoQA candidate datasets...")

required_objects = [
    "train_qa_input_df",
    "validation_qa_input_df",
    "train_clip_text_df",
    "validation_clip_text_df",
    "train_video_representation_df",
    "validation_video_representation_df",
    "clip_text_columns",
    "video_embedding_columns",
    "video_id_column",
    "question_column",
    "ground_truth_answer_column",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for representation QA preparation: "
        + ", ".join(missing_objects)
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} choices, "
        f"found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Define reusable candidate dataset builder
# ------------------------------------------------------------

def build_representation_qa_dataset(
    qa_input_df,
    filtered_clip_text_df,
    filtered_video_representation_df,
    expected_split_name,
):

    qa_input_df = qa_input_df.copy()
    filtered_clip_text_df = filtered_clip_text_df.copy()
    filtered_video_representation_df = filtered_video_representation_df.copy()

    qa_input_df["annotation_id"] = qa_input_df["annotation_id"].astype(str)
    qa_input_df[video_id_column] = qa_input_df[video_id_column].astype(str)

    filtered_clip_text_df["annotation_id"] = (
        filtered_clip_text_df["annotation_id"].astype(str)
    )

    filtered_clip_text_df["choice_index"] = (
        filtered_clip_text_df["choice_index"].astype(int)
    )

    filtered_video_representation_df[video_id_column] = (
        filtered_video_representation_df[video_id_column].astype(str)
    )

    qa_splits = set(qa_input_df["split"].astype(str).unique())

    if qa_splits != {expected_split_name}:
        raise ValueError(
            f"Expected QA split '{expected_split_name}', "
            f"found: {sorted(qa_splits)}"
        )

    text_splits = set(filtered_clip_text_df["split"].astype(str).unique())

    if text_splits != {expected_split_name}:
        raise ValueError(
            f"Expected text representation split '{expected_split_name}', "
            f"found: {sorted(text_splits)}"
        )

    video_splits = set(
        filtered_video_representation_df["split"].astype(str).unique()
    )

    if video_splits != {expected_split_name}:
        raise ValueError(
            f"Expected video representation split '{expected_split_name}', "
            f"found: {sorted(video_splits)}"
        )

    question_answer_text_df = filtered_clip_text_df[
        filtered_clip_text_df["text_type"] == "question_answer"
    ].copy()

    if question_answer_text_df.empty:
        raise RuntimeError(
            "No question-answer text representations were found."
        )

    question_answer_count_by_annotation = (
        question_answer_text_df
        .groupby("annotation_id")
        .size()
    )

    invalid_question_answer_counts = question_answer_count_by_annotation[
        question_answer_count_by_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
    ]

    if len(invalid_question_answer_counts) > 0:
        raise ValueError(
            "Each QA record must have exactly "
            f"{NUM_MULTIPLE_CHOICE_ANSWERS} question-answer representations. "
            f"Invalid records found: {len(invalid_question_answer_counts)}"
        )

    invalid_choice_values = sorted(
        set(question_answer_text_df["choice_index"].unique())
        - set(range(NUM_MULTIPLE_CHOICE_ANSWERS))
    )

    if invalid_choice_values:
        raise ValueError(
            "Found invalid question-answer choice_index values: "
            + str(invalid_choice_values)
        )

    duplicate_question_answer_rows = (
        question_answer_text_df[
            [
                "annotation_id",
                "choice_index",
            ]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_question_answer_rows > 0:
        raise ValueError(
            f"Found {duplicate_question_answer_rows} duplicate "
            "annotation_id/choice_index question-answer representations."
        )

    video_lookup_df = (
        filtered_video_representation_df[
            [
                video_id_column,
                "record_id",
                "representation_source",
            ]
        ]
        .rename(
            columns={
                "record_id": "video_representation_id",
                "representation_source": "video_representation_source",
            }
        )
    )

    unexpected_video_sources = set(
        video_lookup_df["video_representation_source"].astype(str).unique()
    ) - {VIDEO_REPRESENTATION_SOURCE}

    if unexpected_video_sources:
        raise ValueError(
            "Unexpected video representation sources found: "
            + ", ".join(sorted(unexpected_video_sources))
        )

    question_answer_input_df = (
        question_answer_text_df[
            [
                "annotation_id",
                "record_id",
                "choice_index",
                "text",
            ]
        ]
        .rename(
            columns={
                "record_id": "question_answer_representation_id",
                "text": "question_answer_text_from_representation",
            }
        )
        .copy()
    )

    representation_qa_df = (
        qa_input_df
        .merge(video_lookup_df, on=video_id_column, how="left")
        .merge(question_answer_input_df, on="annotation_id", how="left")
    )

    expected_candidate_rows = (
        len(qa_input_df)
        * NUM_MULTIPLE_CHOICE_ANSWERS
    )

    if len(representation_qa_df) != expected_candidate_rows:
        raise ValueError(
            f"Expected {expected_candidate_rows:,} representation QA rows, "
            f"found {len(representation_qa_df):,}."
        )

    required_representation_qa_columns = [
        "qa_record_id",
        "annotation_id",
        "split",
        video_id_column,
        question_column,
        ground_truth_answer_column,
        "video_representation_id",
        "video_representation_source",
        "question_answer_representation_id",
        "choice_index",
        "question_answer_text_from_representation",
    ]

    missing_representation_qa_columns = [
        col for col in required_representation_qa_columns
        if col not in representation_qa_df.columns
    ]

    if missing_representation_qa_columns:
        raise ValueError(
            "representation_qa_df is missing required columns: "
            + ", ".join(missing_representation_qa_columns)
        )

    missing_required_values = (
        representation_qa_df[required_representation_qa_columns]
        .isna()
        .sum()
        .sum()
    )

    if missing_required_values > 0:
        raise ValueError(
            f"Found {missing_required_values} missing required values "
            "in representation_qa_df."
        )

    candidate_count_by_qa = representation_qa_df.groupby("qa_record_id").size()

    invalid_candidate_counts = candidate_count_by_qa[
        candidate_count_by_qa != NUM_MULTIPLE_CHOICE_ANSWERS
    ]

    if len(invalid_candidate_counts) > 0:
        raise ValueError(
            "Each QA record must have exactly "
            f"{NUM_MULTIPLE_CHOICE_ANSWERS} candidate answers. "
            f"Invalid QA records found: {len(invalid_candidate_counts)}"
        )

    duplicate_candidate_rows = (
        representation_qa_df[
            [
                "qa_record_id",
                "choice_index",
            ]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_candidate_rows > 0:
        raise ValueError(
            f"Found {duplicate_candidate_rows} duplicate QA/choice rows."
        )

    return representation_qa_df


# ------------------------------------------------------------
# Build train and validation candidate datasets
# ------------------------------------------------------------

train_representation_qa_df = build_representation_qa_dataset(
    qa_input_df=train_qa_input_df,
    filtered_clip_text_df=train_clip_text_df,
    filtered_video_representation_df=train_video_representation_df,
    expected_split_name="train",
)

validation_representation_qa_df = build_representation_qa_dataset(
    qa_input_df=validation_qa_input_df,
    filtered_clip_text_df=validation_clip_text_df,
    filtered_video_representation_df=validation_video_representation_df,
    expected_split_name=evaluation_split,
)

# Preserve existing downstream validation dataset name
representation_qa_df = validation_representation_qa_df.copy()

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("Representation-based VideoQA candidate datasets prepared successfully.")
print(f"Prediction method          : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Training QA records        : {len(train_qa_input_df):,}")
print(f"Training candidate rows    : {len(train_representation_qa_df):,}")
print(f"Validation QA records      : {len(validation_qa_input_df):,}")
print(f"Validation candidate rows  : {len(validation_representation_qa_df):,}")
print(f"Choices per question       : {NUM_MULTIPLE_CHOICE_ANSWERS}")
print(
    "Training unique videos     : "
    f"{train_representation_qa_df[video_id_column].nunique():,}"
)
print(
    "Validation unique videos   : "
    f"{validation_representation_qa_df[video_id_column].nunique():,}"
)
print(f"Video source               : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text embedding dim         : {len(clip_text_columns):,}")
print(f"Video embedding dim        : {len(video_embedding_columns):,}")

print("\nValidation Representation QA Preview:")
display(
    validation_representation_qa_df[
        [
            "qa_record_id",
            "annotation_id",
            "split",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            "choice_index",
            "question_answer_text_from_representation",
            "video_representation_id",
            "video_representation_source",
            "question_answer_representation_id",
        ]
    ].head(10)
)



### 🔷 Step 8 — Build Grouped Representation QA Samples

* Convert the candidate-level datasets from Step 7 into one grouped sample per QA record.
* Preserve the five ordered question-answer candidate embeddings for each question.
* Attach the corresponding video embedding and ground-truth answer index.
* Build independent training and validation sample collections.
* Validate embedding dimensions, candidate ordering, labels, and required representation identifiers.
* Establish the shared input structure used by all representation-based prediction methods.


In [ ]:
# ============================================================
# Step 8: Build Grouped Representation QA Samples
# ============================================================

import numpy as np

print("Building grouped representation-based QA samples...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "train_representation_qa_df",
    "validation_representation_qa_df",
    "train_clip_text_df",
    "validation_clip_text_df",
    "train_video_representation_df",
    "validation_video_representation_df",
    "clip_text_columns",
    "video_embedding_columns",
    "ground_truth_answer_column",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for grouped QA sample construction: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Validate embedding column configuration
# ------------------------------------------------------------

if not clip_text_columns:
    raise ValueError("No CLIP text embedding columns were defined.")

if not video_embedding_columns:
    raise ValueError("No video embedding columns were defined.")

text_embedding_dimension = len(clip_text_columns)
video_embedding_dimension = len(video_embedding_columns)

if text_embedding_dimension <= 0:
    raise ValueError(
        f"Invalid text embedding dimension: {text_embedding_dimension}"
    )

if video_embedding_dimension <= 0:
    raise ValueError(
        f"Invalid video embedding dimension: {video_embedding_dimension}"
    )

# ------------------------------------------------------------
# Define reusable grouped-sample builder
# ------------------------------------------------------------

def build_representation_qa_samples(
    representation_qa_df,
    clip_text_df,
    video_representation_df,
    expected_split_name,
):
    """
    Convert candidate-level representation rows into grouped
    five-choice VideoQA samples.

    Each returned sample contains:
      - one video embedding,
      - five ordered question-answer embeddings,
      - one ground-truth choice index,
      - QA-level metadata,
      - candidate-level metadata.
    """

    representation_qa_df = representation_qa_df.copy()
    clip_text_df = clip_text_df.copy()
    video_representation_df = video_representation_df.copy()

    # --------------------------------------------------------
    # Validate source datasets
    # --------------------------------------------------------

    if representation_qa_df.empty:
        raise RuntimeError(
            f"No candidate rows were provided for split "
            f"'{expected_split_name}'."
        )

    if clip_text_df.empty:
        raise RuntimeError(
            f"No CLIP text representations were provided for split "
            f"'{expected_split_name}'."
        )

    if video_representation_df.empty:
        raise RuntimeError(
            f"No video representations were provided for split "
            f"'{expected_split_name}'."
        )

    required_candidate_columns = [
        "qa_record_id",
        "split",
        "choice_index",
        "question_answer_representation_id",
        "video_representation_id",
        ground_truth_answer_column,
    ]

    missing_candidate_columns = [
        column
        for column in required_candidate_columns
        if column not in representation_qa_df.columns
    ]

    if missing_candidate_columns:
        raise ValueError(
            "Representation QA dataset is missing required columns: "
            + ", ".join(missing_candidate_columns)
        )

    if "record_id" not in clip_text_df.columns:
        raise ValueError(
            "CLIP text representation dataset is missing 'record_id'."
        )

    if "record_id" not in video_representation_df.columns:
        raise ValueError(
            "Video representation dataset is missing 'record_id'."
        )

    missing_text_embedding_columns = [
        column
        for column in clip_text_columns
        if column not in clip_text_df.columns
    ]

    if missing_text_embedding_columns:
        raise ValueError(
            "CLIP text representation dataset is missing embedding columns: "
            + ", ".join(missing_text_embedding_columns[:10])
        )

    missing_video_embedding_columns = [
        column
        for column in video_embedding_columns
        if column not in video_representation_df.columns
    ]

    if missing_video_embedding_columns:
        raise ValueError(
            "Video representation dataset is missing embedding columns: "
            + ", ".join(missing_video_embedding_columns[:10])
        )

    # --------------------------------------------------------
    # Validate split membership
    # --------------------------------------------------------

    candidate_splits = set(
        representation_qa_df["split"].astype(str).unique()
    )

    if candidate_splits != {expected_split_name}:
        raise ValueError(
            f"Expected candidate split '{expected_split_name}', "
            f"found: {sorted(candidate_splits)}"
        )

    # --------------------------------------------------------
    # Validate representation identifiers
    # --------------------------------------------------------

    if clip_text_df["record_id"].duplicated().any():
        duplicate_ids = (
            clip_text_df.loc[
                clip_text_df["record_id"].duplicated(keep=False),
                "record_id",
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicate CLIP text representation IDs found: "
            + str(duplicate_ids[:10])
        )

    if video_representation_df["record_id"].duplicated().any():
        duplicate_ids = (
            video_representation_df.loc[
                video_representation_df["record_id"].duplicated(keep=False),
                "record_id",
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicate video representation IDs found: "
            + str(duplicate_ids[:10])
        )

    # --------------------------------------------------------
    # Build representation lookup dictionaries
    # --------------------------------------------------------

    text_embedding_lookup = {}

    for row in clip_text_df.to_dict("records"):
        embedding = np.asarray(
            [row[column] for column in clip_text_columns],
            dtype=np.float32,
        )

        if embedding.shape != (text_embedding_dimension,):
            raise ValueError(
                f"Invalid text embedding shape for {row['record_id']}: "
                f"{embedding.shape}"
            )

        if not np.isfinite(embedding).all():
            raise ValueError(
                f"Non-finite text embedding values found for "
                f"{row['record_id']}."
            )

        text_embedding_lookup[str(row["record_id"])] = embedding

    video_embedding_lookup = {}

    for row in video_representation_df.to_dict("records"):
        embedding = np.asarray(
            [row[column] for column in video_embedding_columns],
            dtype=np.float32,
        )

        if embedding.shape != (video_embedding_dimension,):
            raise ValueError(
                f"Invalid video embedding shape for {row['record_id']}: "
                f"{embedding.shape}"
            )

        if not np.isfinite(embedding).all():
            raise ValueError(
                f"Non-finite video embedding values found for "
                f"{row['record_id']}."
            )

        video_embedding_lookup[str(row["record_id"])] = embedding

    # --------------------------------------------------------
    # Build one grouped sample per QA record
    # --------------------------------------------------------

    qa_samples = []

    grouped_candidate_rows = representation_qa_df.groupby(
        "qa_record_id",
        sort=False,
    )

    for qa_record_id, group_df in grouped_candidate_rows:

        group_df = (
            group_df
            .sort_values("choice_index")
            .reset_index(drop=True)
        )

        if len(group_df) != NUM_MULTIPLE_CHOICE_ANSWERS:
            raise ValueError(
                f"QA record {qa_record_id} has {len(group_df)} candidates; "
                f"expected {NUM_MULTIPLE_CHOICE_ANSWERS}."
            )

        actual_choice_indices = group_df["choice_index"].astype(int).tolist()
        expected_choice_indices = list(
            range(NUM_MULTIPLE_CHOICE_ANSWERS)
        )

        if actual_choice_indices != expected_choice_indices:
            raise ValueError(
                f"QA record {qa_record_id} has invalid choice ordering: "
                f"{actual_choice_indices}; "
                f"expected {expected_choice_indices}."
            )

        first_row = group_df.iloc[0]

        ground_truth_choice = int(
            first_row[ground_truth_answer_column]
        )

        if ground_truth_choice not in expected_choice_indices:
            raise ValueError(
                f"Invalid ground-truth choice for {qa_record_id}: "
                f"{ground_truth_choice}"
            )

        ground_truth_values = set(
            group_df[ground_truth_answer_column]
            .astype(int)
            .tolist()
        )

        if ground_truth_values != {ground_truth_choice}:
            raise ValueError(
                f"Inconsistent ground-truth choices for QA record "
                f"{qa_record_id}: {sorted(ground_truth_values)}"
            )

        video_representation_ids = (
            group_df["video_representation_id"]
            .astype(str)
            .unique()
            .tolist()
        )

        if len(video_representation_ids) != 1:
            raise ValueError(
                f"QA record {qa_record_id} references multiple video "
                f"representations: {video_representation_ids}"
            )

        video_representation_id = video_representation_ids[0]

        if video_representation_id not in video_embedding_lookup:
            raise KeyError(
                f"Missing video embedding: {video_representation_id}"
            )

        question_answer_embeddings = []

        for _, candidate_row in group_df.iterrows():

            question_answer_representation_id = str(
                candidate_row[
                    "question_answer_representation_id"
                ]
            )

            if (
                question_answer_representation_id
                not in text_embedding_lookup
            ):
                raise KeyError(
                    "Missing question-answer embedding: "
                    f"{question_answer_representation_id}"
                )

            question_answer_embeddings.append(
                text_embedding_lookup[
                    question_answer_representation_id
                ]
            )

        question_answer_embedding_matrix = np.stack(
            question_answer_embeddings,
            axis=0,
        )

        expected_text_shape = (
            NUM_MULTIPLE_CHOICE_ANSWERS,
            text_embedding_dimension,
        )

        if (
            question_answer_embedding_matrix.shape
            != expected_text_shape
        ):
            raise ValueError(
                f"Invalid question-answer embedding shape for "
                f"{qa_record_id}: "
                f"{question_answer_embedding_matrix.shape}; "
                f"expected {expected_text_shape}."
            )

        qa_samples.append(
            {
                "qa_record_id": str(qa_record_id),
                "metadata": first_row.to_dict(),
                "candidate_metadata": group_df.to_dict("records"),
                "video_embedding": video_embedding_lookup[
                    video_representation_id
                ],
                "question_answer_embeddings":
                    question_answer_embedding_matrix,
                "label": ground_truth_choice,
            }
        )

    if not qa_samples:
        raise RuntimeError(
            f"No grouped QA samples were created for split "
            f"'{expected_split_name}'."
        )

    return qa_samples


# ------------------------------------------------------------
# Build training and validation samples
# ------------------------------------------------------------

train_qa_samples = build_representation_qa_samples(
    representation_qa_df=train_representation_qa_df,
    clip_text_df=train_clip_text_df,
    video_representation_df=train_video_representation_df,
    expected_split_name="train",
)

validation_qa_samples = build_representation_qa_samples(
    representation_qa_df=validation_representation_qa_df,
    clip_text_df=validation_clip_text_df,
    video_representation_df=validation_video_representation_df,
    expected_split_name=evaluation_split,
)

# ------------------------------------------------------------
# Strict record-count validation
# ------------------------------------------------------------

expected_training_qa_records = (
    train_representation_qa_df["qa_record_id"].nunique()
)

expected_validation_qa_records = (
    validation_representation_qa_df["qa_record_id"].nunique()
)

if len(train_qa_samples) != expected_training_qa_records:
    raise ValueError(
        f"Expected {expected_training_qa_records:,} grouped training "
        f"QA samples, found {len(train_qa_samples):,}."
    )

if len(validation_qa_samples) != expected_validation_qa_records:
    raise ValueError(
        f"Expected {expected_validation_qa_records:,} grouped validation "
        f"QA samples, found {len(validation_qa_samples):,}."
    )

# ------------------------------------------------------------
# Initialize shared method-output contract
# ------------------------------------------------------------

all_scores = None
fusion_training_history_df = None
training_epochs = None
final_training_loss = None
fusion_embedding_dimension = None

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("Grouped representation-based QA samples created successfully.")
print(f"Training QA samples      : {len(train_qa_samples):,}")
print(f"Validation QA samples    : {len(validation_qa_samples):,}")
print(f"Choices per QA sample    : {NUM_MULTIPLE_CHOICE_ANSWERS}")
print(f"Text embedding dimension : {text_embedding_dimension}")
print(f"Video embedding dimension: {video_embedding_dimension}")
print("Shared method outputs    : initialized")



### 🔷 Step 9 — Run Cosine-Similarity Scoring

* Execute cosine-similarity scoring when `REPRESENTATION_VIDEOQA_METHOD` is set to `cosine_similarity`.
* Require matching video and question-answer embedding dimensions.
* Normalize each validation video embedding and its five candidate text embeddings.
* Calculate one cosine-similarity score for each answer choice.
* Produce the shared score matrix consumed by the standardized prediction-generation step.
* Skip this step without modifying shared outputs when another prediction method is selected.



In [ ]:
# ============================================================
# Step 9: Run Cosine-Similarity Scoring
# ============================================================

import numpy as np
import pandas as pd

print("Evaluating cosine-similarity scoring step...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for cosine-similarity scoring: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Run cosine-similarity scoring when selected
# ------------------------------------------------------------

if REPRESENTATION_VIDEOQA_METHOD == "cosine_similarity":

    print("Running cosine-similarity scoring...")

    # --------------------------------------------------------
    # Validate compatible embedding dimensions
    # --------------------------------------------------------

    if video_embedding_dimension != text_embedding_dimension:
        raise ValueError(
            "cosine_similarity requires matching video and text "
            "embedding dimensions. "
            f"Video dim={video_embedding_dimension}, "
            f"text dim={text_embedding_dimension}."
        )

    if not validation_qa_samples:
        raise RuntimeError(
            "No validation QA samples are available for "
            "cosine-similarity scoring."
        )

    # --------------------------------------------------------
    # Define normalization helper
    # --------------------------------------------------------

    def normalize_embedding_matrix(embedding_matrix):
        """
        Normalize each embedding vector to unit length.
        """

        embedding_matrix = np.asarray(
            embedding_matrix,
            dtype=np.float32,
        )

        if embedding_matrix.ndim != 2:
            raise ValueError(
                "Expected a two-dimensional embedding matrix, "
                f"found shape {embedding_matrix.shape}."
            )

        if not np.isfinite(embedding_matrix).all():
            raise ValueError(
                "Non-finite embedding values encountered during "
                "cosine-similarity scoring."
            )

        norms = np.linalg.norm(
            embedding_matrix,
            axis=1,
            keepdims=True,
        )

        if np.any(norms == 0):
            raise ValueError(
                "Zero-norm embedding encountered during "
                "cosine-similarity scoring."
            )

        return embedding_matrix / norms

    # --------------------------------------------------------
    # Score the five answer choices for each validation record
    # --------------------------------------------------------

    cosine_score_rows = []

    for sample in validation_qa_samples:

        video_embedding = np.asarray(
            sample["video_embedding"],
            dtype=np.float32,
        ).reshape(1, -1)

        question_answer_embeddings = np.asarray(
            sample["question_answer_embeddings"],
            dtype=np.float32,
        )

        expected_video_shape = (
            1,
            video_embedding_dimension,
        )

        expected_text_shape = (
            NUM_MULTIPLE_CHOICE_ANSWERS,
            text_embedding_dimension,
        )

        if video_embedding.shape != expected_video_shape:
            raise ValueError(
                f"Invalid video embedding shape for "
                f"{sample['qa_record_id']}: "
                f"{video_embedding.shape}; "
                f"expected {expected_video_shape}."
            )

        if question_answer_embeddings.shape != expected_text_shape:
            raise ValueError(
                f"Invalid question-answer embedding shape for "
                f"{sample['qa_record_id']}: "
                f"{question_answer_embeddings.shape}; "
                f"expected {expected_text_shape}."
            )

        normalized_video_embedding = normalize_embedding_matrix(
            video_embedding
        )

        normalized_question_answer_embeddings = (
            normalize_embedding_matrix(
                question_answer_embeddings
            )
        )

        cosine_scores = (
            normalized_question_answer_embeddings
            @ normalized_video_embedding.T
        ).reshape(-1)

        if cosine_scores.shape != (
            NUM_MULTIPLE_CHOICE_ANSWERS,
        ):
            raise ValueError(
                f"Invalid cosine score shape for "
                f"{sample['qa_record_id']}: "
                f"{cosine_scores.shape}."
            )

        if not np.isfinite(cosine_scores).all():
            raise ValueError(
                f"Non-finite cosine scores generated for "
                f"{sample['qa_record_id']}."
            )

        cosine_score_rows.append(cosine_scores)

    # --------------------------------------------------------
    # Populate shared method outputs
    # --------------------------------------------------------

    all_scores = np.vstack(cosine_score_rows)

    fusion_training_history_df = pd.DataFrame(
        [
            {
                "epoch": 0,
                "loss": np.nan,
            }
        ]
    )

    training_epochs = 0
    final_training_loss = np.nan
    fusion_embedding_dimension = 0

    # --------------------------------------------------------
    # Strict output validation
    # --------------------------------------------------------

    expected_score_shape = (
        len(validation_qa_samples),
        NUM_MULTIPLE_CHOICE_ANSWERS,
    )

    if all_scores.shape != expected_score_shape:
        raise ValueError(
            f"Expected cosine score matrix shape "
            f"{expected_score_shape}, found {all_scores.shape}."
        )

    if not np.isfinite(all_scores).all():
        raise ValueError(
            "The cosine-similarity score matrix contains "
            "non-finite values."
        )

    if np.any(all_scores < -1.000001) or np.any(
        all_scores > 1.000001
    ):
        raise ValueError(
            "Cosine-similarity scores were generated outside "
            "the expected range [-1, 1]."
        )

    print("Cosine-similarity scoring completed successfully.")
    print(f"Validation QA records : {len(validation_qa_samples):,}")
    print(f"Choices per QA record : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Score matrix shape    : {all_scores.shape}")
    print(f"Minimum score         : {all_scores.min():.6f}")
    print(f"Maximum score         : {all_scores.max():.6f}")
    print("Training epochs       : not applicable")
    print("Training loss         : not applicable")

# ------------------------------------------------------------
# Skip when another prediction method is selected
# ------------------------------------------------------------

else:
    print(
        "Cosine-similarity scoring skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 10 — Run Fusion MLP Classifier

* Execute the Fusion MLP classifier when `REPRESENTATION_VIDEOQA_METHOD` is set to `fusion_mlp_classifier`.
* Use the grouped training and validation QA samples created in Step 8.
* Project video and question-answer embeddings into a shared fusion space.
* Concatenate the projected video and candidate text representations.
* Train a grouped five-choice classifier using cross-entropy loss.
* Produce the shared validation score matrix consumed by the standardized prediction-generation step.
* Skip this step without modifying shared outputs when another prediction method is selected.



In [ ]:
# ============================================================
# Step 10: Run Fusion MLP Classifier
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Evaluating Fusion MLP classifier step...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "train_qa_samples",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for Fusion MLP classification: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Run Fusion MLP classifier when selected
# ------------------------------------------------------------

if REPRESENTATION_VIDEOQA_METHOD == "fusion_mlp_classifier":

    print("Running Fusion MLP classifier...")

    # --------------------------------------------------------
    # Reproducibility and runtime device
    # --------------------------------------------------------

    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # --------------------------------------------------------
    # Validate shared grouped samples
    # --------------------------------------------------------

    if not train_qa_samples:
        raise RuntimeError(
            "No training QA samples are available for Fusion MLP training."
        )

    if not validation_qa_samples:
        raise RuntimeError(
            "No validation QA samples are available for Fusion MLP scoring."
        )

    # --------------------------------------------------------
    # Define grouped QA dataset
    # --------------------------------------------------------

    class FusionQADataset(Dataset):

        def __init__(self, samples):
            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, index):

            sample = self.samples[index]

            video_embedding = np.asarray(
                sample["video_embedding"],
                dtype=np.float32,
            )

            question_answer_embeddings = np.asarray(
                sample["question_answer_embeddings"],
                dtype=np.float32,
            )

            label = int(sample["label"])

            expected_video_shape = (
                video_embedding_dimension,
            )

            expected_text_shape = (
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if video_embedding.shape != expected_video_shape:
                raise ValueError(
                    f"Invalid video embedding shape for "
                    f"{sample['qa_record_id']}: "
                    f"{video_embedding.shape}; "
                    f"expected {expected_video_shape}."
                )

            if question_answer_embeddings.shape != expected_text_shape:
                raise ValueError(
                    f"Invalid question-answer embedding shape for "
                    f"{sample['qa_record_id']}: "
                    f"{question_answer_embeddings.shape}; "
                    f"expected {expected_text_shape}."
                )

            if label not in range(NUM_MULTIPLE_CHOICE_ANSWERS):
                raise ValueError(
                    f"Invalid label for {sample['qa_record_id']}: "
                    f"{label}"
                )

            if not np.isfinite(video_embedding).all():
                raise ValueError(
                    f"Non-finite video embedding values found for "
                    f"{sample['qa_record_id']}."
                )

            if not np.isfinite(question_answer_embeddings).all():
                raise ValueError(
                    f"Non-finite text embedding values found for "
                    f"{sample['qa_record_id']}."
                )

            return {
                "video": torch.tensor(
                    video_embedding,
                    dtype=torch.float32,
                ),
                "question_answers": torch.tensor(
                    question_answer_embeddings,
                    dtype=torch.float32,
                ),
                "label": torch.tensor(
                    label,
                    dtype=torch.long,
                ),
            }

    # --------------------------------------------------------
    # Build training and validation data loaders
    # --------------------------------------------------------

    train_fusion_dataset = FusionQADataset(
        train_qa_samples
    )

    validation_fusion_dataset = FusionQADataset(
        validation_qa_samples
    )

    train_fusion_loader = DataLoader(
        train_fusion_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=True,
    )

    validation_fusion_loader = DataLoader(
        validation_fusion_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=False,
    )

    # --------------------------------------------------------
    # Define Fusion MLP classifier
    # --------------------------------------------------------

    class FusionMLPClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            fusion_input_dim = fusion_embedding_dim * 2

            layers = [
                nn.Linear(
                    fusion_input_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(hidden_dim_1)
                )

            layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(hidden_dim_2)
                )

            layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(*layers)

        def forward(
            self,
            video,
            question_answers,
        ):

            batch_size, num_choices, _ = (
                question_answers.shape
            )

            video_projected = self.video_projection(
                video
            )

            question_answers_projected = (
                self.text_projection(
                    question_answers
                )
            )

            video_expanded = (
                video_projected
                .unsqueeze(1)
                .expand(
                    -1,
                    num_choices,
                    -1,
                )
            )

            fused = torch.cat(
                [
                    video_expanded,
                    question_answers_projected,
                ],
                dim=-1,
            )

            scores = (
                self.scorer(fused)
                .squeeze(-1)
            )

            return scores

    # --------------------------------------------------------
    # Initialize model, loss, and optimizer
    # --------------------------------------------------------

    fusion_model = FusionMLPClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        fusion_model.parameters(),
        lr=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
    )

    # --------------------------------------------------------
    # Train Fusion MLP classifier
    # --------------------------------------------------------

    fusion_model.train()

    training_history = []

    for epoch in range(FUSION_EPOCHS):

        epoch_losses = []

        for batch in train_fusion_loader:

            video_batch = batch["video"].to(device)

            question_answer_batch = (
                batch["question_answers"]
                .to(device)
            )

            label_batch = batch["label"].to(device)

            optimizer.zero_grad()

            scores = fusion_model(
                video_batch,
                question_answer_batch,
            )

            expected_score_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if scores.shape != expected_score_shape:
                raise ValueError(
                    f"Unexpected training score shape: "
                    f"{scores.shape}; "
                    f"expected {expected_score_shape}."
                )

            loss = criterion(
                scores,
                label_batch,
            )

            if not torch.isfinite(loss):
                raise ValueError(
                    "Non-finite loss encountered during "
                    "Fusion MLP training."
                )

            loss.backward()
            optimizer.step()

            epoch_losses.append(
                float(loss.item())
            )

        if not epoch_losses:
            raise RuntimeError(
                f"No training batches were processed "
                f"during epoch {epoch + 1}."
            )

        mean_epoch_loss = float(
            np.mean(epoch_losses)
        )

        training_history.append(
            {
                "epoch": epoch + 1,
                "loss": mean_epoch_loss,
            }
        )

    fusion_training_history_df = pd.DataFrame(
        training_history
    )

    # --------------------------------------------------------
    # Generate validation score matrix
    # --------------------------------------------------------

    fusion_model.eval()

    validation_score_batches = []

    with torch.no_grad():

        for batch in validation_fusion_loader:

            video_batch = batch["video"].to(device)

            question_answer_batch = (
                batch["question_answers"]
                .to(device)
            )

            scores = fusion_model(
                video_batch,
                question_answer_batch,
            )

            expected_score_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if scores.shape != expected_score_shape:
                raise ValueError(
                    f"Unexpected validation score shape: "
                    f"{scores.shape}; "
                    f"expected {expected_score_shape}."
                )

            if not torch.isfinite(scores).all():
                raise ValueError(
                    "Non-finite validation scores generated "
                    "by the Fusion MLP classifier."
                )

            validation_score_batches.append(
                scores.cpu().numpy()
            )

    if not validation_score_batches:
        raise RuntimeError(
            "No validation batches were scored by the "
            "Fusion MLP classifier."
        )

    # --------------------------------------------------------
    # Populate shared method outputs
    # --------------------------------------------------------

    all_scores = np.vstack(
        validation_score_batches
    )

    training_epochs = FUSION_EPOCHS

    final_training_loss = float(
        fusion_training_history_df[
            "loss"
        ].iloc[-1]
    )

    fusion_embedding_dimension = (
        FUSION_EMBEDDING_DIM
    )

    # --------------------------------------------------------
    # Strict output validation
    # --------------------------------------------------------

    expected_all_scores_shape = (
        len(validation_qa_samples),
        NUM_MULTIPLE_CHOICE_ANSWERS,
    )

    if all_scores.shape != expected_all_scores_shape:
        raise ValueError(
            f"Expected Fusion MLP score matrix shape "
            f"{expected_all_scores_shape}, "
            f"found {all_scores.shape}."
        )

    if not np.isfinite(all_scores).all():
        raise ValueError(
            "Fusion MLP score matrix contains "
            "non-finite values."
        )

    if len(fusion_training_history_df) != FUSION_EPOCHS:
        raise ValueError(
            f"Expected {FUSION_EPOCHS} training history rows, "
            f"found {len(fusion_training_history_df)}."
        )

    # --------------------------------------------------------
    # Display summary
    # --------------------------------------------------------

    print("Fusion MLP classifier completed successfully.")
    print(f"Device                    : {device}")
    print(f"Training QA records       : {len(train_qa_samples):,}")
    print(f"Validation QA records     : {len(validation_qa_samples):,}")
    print(f"Choices per QA record     : {NUM_MULTIPLE_CHOICE_ANSWERS}")
    print(f"Text embedding dimension  : {text_embedding_dimension}")
    print(f"Video embedding dimension : {video_embedding_dimension}")
    print(f"Fusion embedding dimension: {fusion_embedding_dimension}")
    print(f"Training epochs           : {training_epochs}")
    print(f"Final training loss       : {final_training_loss:.4f}")
    print(f"Score matrix shape        : {all_scores.shape}")

# ------------------------------------------------------------
# Skip when another prediction method is selected
# ------------------------------------------------------------

else:
    print(
        "Fusion MLP classifier skipped. "
        f"Selected method: {REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 11 — Run Interaction Fusion Classifier

* Execute the interaction-fusion classifier when `REPRESENTATION_VIDEOQA_METHOD` is set to `interaction_fusion_classifier`.
* Use the grouped training and validation QA samples created in Step 8.
* Project video and question-answer embeddings into a shared fusion space.
* Construct explicit interaction features using the projected embeddings, absolute differences, and elementwise products.
* Train a grouped five-choice classifier using cross-entropy loss.
* Produce the shared validation score matrix consumed by Step 14.
* Skip this step without modifying shared outputs when another prediction method is selected.



In [ ]:
# ============================================================
# Step 11: Run Interaction Fusion Classifier
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Evaluating interaction fusion classifier step...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "REPRESENTATION_VIDEOQA_METHOD",
    "train_qa_samples",
    "validation_qa_samples",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for interaction fusion classification: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Run interaction fusion classifier when selected
# ------------------------------------------------------------

if (
    REPRESENTATION_VIDEOQA_METHOD
    == "interaction_fusion_classifier"
):

    print("Running interaction fusion classifier...")

    # --------------------------------------------------------
    # Reproducibility and runtime device
    # --------------------------------------------------------

    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            FUSION_RANDOM_SEED
        )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    # --------------------------------------------------------
    # Validate grouped samples
    # --------------------------------------------------------

    if not train_qa_samples:
        raise RuntimeError(
            "No training QA samples are available for "
            "interaction fusion training."
        )

    if not validation_qa_samples:
        raise RuntimeError(
            "No validation QA samples are available for "
            "interaction fusion scoring."
        )

    # --------------------------------------------------------
    # Define grouped QA dataset
    # --------------------------------------------------------

    class InteractionFusionQADataset(Dataset):

        def __init__(self, samples):
            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, index):

            sample = self.samples[index]

            video_embedding = np.asarray(
                sample["video_embedding"],
                dtype=np.float32,
            )

            question_answer_embeddings = np.asarray(
                sample["question_answer_embeddings"],
                dtype=np.float32,
            )

            label = int(
                sample["label"]
            )

            expected_video_shape = (
                video_embedding_dimension,
            )

            expected_text_shape = (
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                video_embedding.shape
                != expected_video_shape
            ):
                raise ValueError(
                    f"Invalid video embedding shape for "
                    f"{sample['qa_record_id']}: "
                    f"{video_embedding.shape}; "
                    f"expected {expected_video_shape}."
                )

            if (
                question_answer_embeddings.shape
                != expected_text_shape
            ):
                raise ValueError(
                    f"Invalid question-answer embedding shape for "
                    f"{sample['qa_record_id']}: "
                    f"{question_answer_embeddings.shape}; "
                    f"expected {expected_text_shape}."
                )

            if label not in range(
                NUM_MULTIPLE_CHOICE_ANSWERS
            ):
                raise ValueError(
                    f"Invalid label for "
                    f"{sample['qa_record_id']}: "
                    f"{label}"
                )

            if not np.isfinite(
                video_embedding
            ).all():
                raise ValueError(
                    f"Non-finite video embedding values found for "
                    f"{sample['qa_record_id']}."
                )

            if not np.isfinite(
                question_answer_embeddings
            ).all():
                raise ValueError(
                    f"Non-finite text embedding values found for "
                    f"{sample['qa_record_id']}."
                )

            return {
                "video": torch.tensor(
                    video_embedding,
                    dtype=torch.float32,
                ),
                "question_answers": torch.tensor(
                    question_answer_embeddings,
                    dtype=torch.float32,
                ),
                "label": torch.tensor(
                    label,
                    dtype=torch.long,
                ),
            }

    # --------------------------------------------------------
    # Build data loaders
    # --------------------------------------------------------

    train_interaction_dataset = (
        InteractionFusionQADataset(
            train_qa_samples
        )
    )

    validation_interaction_dataset = (
        InteractionFusionQADataset(
            validation_qa_samples
        )
    )

    train_interaction_loader = DataLoader(
        train_interaction_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=True,
    )

    validation_interaction_loader = DataLoader(
        validation_interaction_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=False,
    )

    # --------------------------------------------------------
    # Define interaction fusion classifier
    # --------------------------------------------------------

    class InteractionFusionClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            interaction_input_dim = (
                fusion_embedding_dim * 4
            )

            layers = [
                nn.Linear(
                    interaction_input_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(
                        hidden_dim_1
                    )
                )

            layers.extend(
                [
                    nn.Dropout(
                        dropout
                    ),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                layers.append(
                    nn.LayerNorm(
                        hidden_dim_2
                    )
                )

            layers.extend(
                [
                    nn.Dropout(
                        dropout
                    ),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(
                *layers
            )

        def forward(
            self,
            video,
            question_answers,
        ):

            _, num_choices, _ = (
                question_answers.shape
            )

            video_projected = (
                self.video_projection(
                    video
                )
            )

            question_answers_projected = (
                self.text_projection(
                    question_answers
                )
            )

            video_expanded = (
                video_projected
                .unsqueeze(1)
                .expand(
                    -1,
                    num_choices,
                    -1,
                )
            )

            absolute_difference = torch.abs(
                video_expanded
                - question_answers_projected
            )

            elementwise_product = (
                video_expanded
                * question_answers_projected
            )

            interaction_features = torch.cat(
                [
                    video_expanded,
                    question_answers_projected,
                    absolute_difference,
                    elementwise_product,
                ],
                dim=-1,
            )

            scores = (
                self.scorer(
                    interaction_features
                )
                .squeeze(-1)
            )

            return scores

    # --------------------------------------------------------
    # Initialize model, loss, and optimizer
    # --------------------------------------------------------

    interaction_model = (
        InteractionFusionClassifier(
            video_input_dim=
                video_embedding_dimension,
            text_input_dim=
                text_embedding_dimension,
            fusion_embedding_dim=
                FUSION_EMBEDDING_DIM,
            hidden_dim_1=
                FUSION_HIDDEN_DIM_1,
            hidden_dim_2=
                FUSION_HIDDEN_DIM_2,
            output_dim=
                FUSION_OUTPUT_DIM,
            dropout=
                FUSION_DROPOUT,
            use_layer_norm=
                USE_LAYER_NORMALIZATION,
        )
        .to(device)
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        interaction_model.parameters(),
        lr=FUSION_LEARNING_RATE,
        weight_decay=
            FUSION_WEIGHT_DECAY,
    )

    # --------------------------------------------------------
    # Train interaction fusion classifier
    # --------------------------------------------------------

    interaction_model.train()

    training_history = []

    for epoch in range(
        FUSION_EPOCHS
    ):

        epoch_losses = []

        for batch in (
            train_interaction_loader
        ):

            video_batch = (
                batch["video"]
                .to(device)
            )

            question_answer_batch = (
                batch["question_answers"]
                .to(device)
            )

            label_batch = (
                batch["label"]
                .to(device)
            )

            optimizer.zero_grad()

            scores = interaction_model(
                video_batch,
                question_answer_batch,
            )

            expected_score_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if (
                scores.shape
                != expected_score_shape
            ):
                raise ValueError(
                    "Unexpected interaction-fusion "
                    f"training score shape: "
                    f"{scores.shape}; "
                    f"expected "
                    f"{expected_score_shape}."
                )

            loss = criterion(
                scores,
                label_batch,
            )

            if not torch.isfinite(
                loss
            ):
                raise ValueError(
                    "Non-finite loss encountered during "
                    "interaction fusion training."
                )

            loss.backward()
            optimizer.step()

            epoch_losses.append(
                float(
                    loss.item()
                )
            )

        if not epoch_losses:
            raise RuntimeError(
                f"No training batches were processed "
                f"during epoch {epoch + 1}."
            )

        mean_epoch_loss = float(
            np.mean(
                epoch_losses
            )
        )

        training_history.append(
            {
                "epoch": epoch + 1,
                "loss": mean_epoch_loss,
            }
        )

    fusion_training_history_df = (
        pd.DataFrame(
            training_history
        )
    )

    # --------------------------------------------------------
    # Generate validation score matrix
    # --------------------------------------------------------

    interaction_model.eval()

    validation_score_batches = []

    with torch.no_grad():

        for batch in (
            validation_interaction_loader
        ):

            video_batch = (
                batch["video"]
                .to(device)
            )

            question_answer_batch = (
                batch["question_answers"]
                .to(device)
            )

            scores = interaction_model(
                video_batch,
                question_answer_batch,
            )

            expected_score_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if (
                scores.shape
                != expected_score_shape
            ):
                raise ValueError(
                    "Unexpected interaction-fusion "
                    f"validation score shape: "
                    f"{scores.shape}; "
                    f"expected "
                    f"{expected_score_shape}."
                )

            if not torch.isfinite(
                scores
            ).all():
                raise ValueError(
                    "Non-finite validation scores "
                    "generated by the interaction "
                    "fusion classifier."
                )

            validation_score_batches.append(
                scores
                .cpu()
                .numpy()
            )

    if not validation_score_batches:
        raise RuntimeError(
            "No validation batches were scored by "
            "the interaction fusion classifier."
        )

    # --------------------------------------------------------
    # Populate shared method outputs
    # --------------------------------------------------------

    all_scores = np.vstack(
        validation_score_batches
    )

    training_epochs = (
        FUSION_EPOCHS
    )

    final_training_loss = float(
        fusion_training_history_df[
            "loss"
        ].iloc[-1]
    )

    fusion_embedding_dimension = (
        FUSION_EMBEDDING_DIM
    )

    # --------------------------------------------------------
    # Strict output validation
    # --------------------------------------------------------

    expected_all_scores_shape = (
        len(
            validation_qa_samples
        ),
        NUM_MULTIPLE_CHOICE_ANSWERS,
    )

    if (
        all_scores.shape
        != expected_all_scores_shape
    ):
        raise ValueError(
            "Expected interaction-fusion score "
            f"matrix shape "
            f"{expected_all_scores_shape}, "
            f"found {all_scores.shape}."
        )

    if not np.isfinite(
        all_scores
    ).all():
        raise ValueError(
            "Interaction-fusion score matrix "
            "contains non-finite values."
        )

    if len(
        fusion_training_history_df
    ) != FUSION_EPOCHS:
        raise ValueError(
            f"Expected {FUSION_EPOCHS} "
            "training-history rows, "
            f"found "
            f"{len(fusion_training_history_df)}."
        )

    # --------------------------------------------------------
    # Display summary
    # --------------------------------------------------------

    print(
        "Interaction fusion classifier "
        "completed successfully."
    )

    print(
        f"Device                    : "
        f"{device}"
    )

    print(
        f"Training QA records       : "
        f"{len(train_qa_samples):,}"
    )

    print(
        f"Validation QA records     : "
        f"{len(validation_qa_samples):,}"
    )

    print(
        f"Choices per QA record     : "
        f"{NUM_MULTIPLE_CHOICE_ANSWERS}"
    )

    print(
        f"Text embedding dimension  : "
        f"{text_embedding_dimension}"
    )

    print(
        f"Video embedding dimension : "
        f"{video_embedding_dimension}"
    )

    print(
        f"Fusion embedding dimension: "
        f"{fusion_embedding_dimension}"
    )

    print(
        f"Interaction feature dim   : "
        f"{fusion_embedding_dimension * 4}"
    )

    print(
        f"Training epochs           : "
        f"{training_epochs}"
    )

    print(
        f"Final training loss       : "
        f"{final_training_loss:.4f}"
    )

    print(
        f"Score matrix shape        : "
        f"{all_scores.shape}"
    )

# ------------------------------------------------------------
# Skip when another prediction method is selected
# ------------------------------------------------------------

else:
    print(
        "Interaction fusion classifier skipped. "
        f"Selected method: "
        f"{REPRESENTATION_VIDEOQA_METHOD}"
    )



### 🔷 Step 12 — Gated Fusion



In [ ]:
# ============================================================
# Step 12: Run Gated Fusion Classifier
# ============================================================

print("Evaluating Gated Fusion classifier method selection.")

if REPRESENTATION_VIDEOQA_METHOD == "gated_fusion_classifier":

    # ------------------------------------------------------------
    # Define computation device when not already available
    # ------------------------------------------------------------

    if "torch" not in globals():
        raise NameError(
            "PyTorch must be imported before running Step 12."
        )

    if "device" not in globals():
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    if not isinstance(device, torch.device):
        raise TypeError(
            "device must be a torch.device object. "
            f"Found {type(device).__name__}."
        )

    print(f"Computation device selected: {device}")

    # ------------------------------------------------------------
    # Verify required inputs
    # ------------------------------------------------------------

    required_gated_fusion_objects = [
        "train_qa_samples",
        "validation_qa_samples",
        "device",
        "text_embedding_dimension",
        "video_embedding_dimension",
        "NUM_MULTIPLE_CHOICE_ANSWERS",
        "FUSION_EMBEDDING_DIM",
        "FUSION_HIDDEN_DIM_1",
        "FUSION_HIDDEN_DIM_2",
        "FUSION_OUTPUT_DIM",
        "FUSION_DROPOUT",
        "FUSION_BATCH_SIZE",
        "FUSION_LEARNING_RATE",
        "FUSION_WEIGHT_DECAY",
        "FUSION_EPOCHS",
        "FUSION_RANDOM_SEED",
        "USE_LAYER_NORMALIZATION",
    ]

    missing_gated_fusion_objects = [
        name
        for name in required_gated_fusion_objects
        if name not in globals()
    ]

    if missing_gated_fusion_objects:
        raise NameError(
            "Missing required objects for Gated Fusion classification: "
            + ", ".join(missing_gated_fusion_objects)
        )

    if not isinstance(train_qa_samples, list) or not train_qa_samples:
        raise RuntimeError(
            "train_qa_samples must be a non-empty list."
        )

    if (
        not isinstance(validation_qa_samples, list)
        or not validation_qa_samples
    ):
        raise RuntimeError(
            "validation_qa_samples must be a non-empty list."
        )

    if text_embedding_dimension <= 0:
        raise ValueError(
            "text_embedding_dimension must be greater than zero."
        )

    if video_embedding_dimension <= 0:
        raise ValueError(
            "video_embedding_dimension must be greater than zero."
        )

    if NUM_MULTIPLE_CHOICE_ANSWERS <= 1:
        raise ValueError(
            "NUM_MULTIPLE_CHOICE_ANSWERS must be greater than one."
        )

    if FUSION_EMBEDDING_DIM <= 0:
        raise ValueError(
            "FUSION_EMBEDDING_DIM must be greater than zero."
        )

    if FUSION_HIDDEN_DIM_1 <= 0:
        raise ValueError(
            "FUSION_HIDDEN_DIM_1 must be greater than zero."
        )

    if FUSION_HIDDEN_DIM_2 <= 0:
        raise ValueError(
            "FUSION_HIDDEN_DIM_2 must be greater than zero."
        )

    if FUSION_OUTPUT_DIM != 1:
        raise ValueError(
            "Gated Fusion requires FUSION_OUTPUT_DIM=1 so that each "
            "candidate answer receives one scalar score. "
            f"Found FUSION_OUTPUT_DIM={FUSION_OUTPUT_DIM}."
        )

    if not 0.0 <= FUSION_DROPOUT < 1.0:
        raise ValueError(
            "FUSION_DROPOUT must be in the range [0.0, 1.0). "
            f"Found {FUSION_DROPOUT}."
        )

    if FUSION_BATCH_SIZE <= 0:
        raise ValueError(
            "FUSION_BATCH_SIZE must be greater than zero."
        )

    if FUSION_LEARNING_RATE <= 0:
        raise ValueError(
            "FUSION_LEARNING_RATE must be greater than zero."
        )

    if FUSION_WEIGHT_DECAY < 0:
        raise ValueError(
            "FUSION_WEIGHT_DECAY cannot be negative."
        )

    if FUSION_EPOCHS <= 0:
        raise ValueError(
            "FUSION_EPOCHS must be greater than zero."
        )

    # ------------------------------------------------------------
    # Validate grouped QA sample structure
    # ------------------------------------------------------------

    def validate_gated_fusion_samples(samples, sample_group_name):

        required_sample_fields = {
            "qa_record_id",
            "video_embedding",
            "question_answer_embeddings",
            "label",
        }

        for sample_index, sample in enumerate(samples):

            if not isinstance(sample, dict):
                raise TypeError(
                    f"{sample_group_name} sample {sample_index} must be "
                    "a dictionary."
                )

            missing_sample_fields = sorted(
                required_sample_fields - set(sample.keys())
            )

            if missing_sample_fields:
                raise KeyError(
                    f"{sample_group_name} sample {sample_index} is missing "
                    "required fields: "
                    + ", ".join(missing_sample_fields)
                )

            video_embedding = np.asarray(
                sample["video_embedding"],
                dtype=np.float32,
            )

            question_answer_embeddings = np.asarray(
                sample["question_answer_embeddings"],
                dtype=np.float32,
            )

            if video_embedding.ndim != 1:
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} video "
                    "embedding must be one-dimensional. "
                    f"Found shape {video_embedding.shape}."
                )

            if (
                video_embedding.shape[0]
                != video_embedding_dimension
            ):
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} video "
                    "embedding dimension mismatch. Expected "
                    f"{video_embedding_dimension}, found "
                    f"{video_embedding.shape[0]}."
                )

            expected_question_answer_shape = (
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                question_answer_embeddings.shape
                != expected_question_answer_shape
            ):
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} "
                    "question-answer embedding shape mismatch. Expected "
                    f"{expected_question_answer_shape}, found "
                    f"{question_answer_embeddings.shape}."
                )

            label = sample["label"]

            if isinstance(label, bool):
                raise TypeError(
                    f"{sample_group_name} sample {sample_index} label "
                    "cannot be boolean."
                )

            try:
                label = int(label)
            except (TypeError, ValueError) as exc:
                raise TypeError(
                    f"{sample_group_name} sample {sample_index} has a "
                    f"non-integer label: {label}"
                ) from exc

            if label not in range(NUM_MULTIPLE_CHOICE_ANSWERS):
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} has "
                    f"invalid label {label}."
                )

            if not np.isfinite(video_embedding).all():
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} contains "
                    "non-finite video embedding values."
                )

            if not np.isfinite(question_answer_embeddings).all():
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} contains "
                    "non-finite question-answer embedding values."
                )


    validate_gated_fusion_samples(
        samples=train_qa_samples,
        sample_group_name="Training",
    )

    validate_gated_fusion_samples(
        samples=validation_qa_samples,
        sample_group_name="Validation",
    )

    # ------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------

    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(FUSION_RANDOM_SEED)
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    data_loader_generator = torch.Generator()
    data_loader_generator.manual_seed(FUSION_RANDOM_SEED)

    # ------------------------------------------------------------
    # Define grouped QA dataset
    # ------------------------------------------------------------

    class GatedFusionQADataset(Dataset):

        def __init__(self, samples):

            if not samples:
                raise RuntimeError(
                    "GatedFusionQADataset received no samples."
                )

            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):

            sample = self.samples[idx]

            return {
                "video": torch.tensor(
                    sample["video_embedding"],
                    dtype=torch.float32,
                ),
                "question_answers": torch.tensor(
                    sample["question_answer_embeddings"],
                    dtype=torch.float32,
                ),
                "label": torch.tensor(
                    int(sample["label"]),
                    dtype=torch.long,
                ),
            }

    # ------------------------------------------------------------
    # Build training and validation data loaders
    # ------------------------------------------------------------

    train_gated_fusion_dataset = GatedFusionQADataset(
        train_qa_samples
    )

    validation_gated_fusion_dataset = GatedFusionQADataset(
        validation_qa_samples
    )

    train_gated_fusion_loader = DataLoader(
        train_gated_fusion_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=True,
        generator=data_loader_generator,
        drop_last=False,
    )

    validation_gated_fusion_loader = DataLoader(
        validation_gated_fusion_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )

    if len(train_gated_fusion_loader) == 0:
        raise RuntimeError(
            "The Gated Fusion training data loader contains no batches."
        )

    if len(validation_gated_fusion_loader) == 0:
        raise RuntimeError(
            "The Gated Fusion validation data loader contains no batches."
        )

    # ------------------------------------------------------------
    # Define Gated Fusion classifier
    # ------------------------------------------------------------

    class GatedFusionClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            if video_input_dim <= 0:
                raise ValueError(
                    "video_input_dim must be greater than zero."
                )

            if text_input_dim <= 0:
                raise ValueError(
                    "text_input_dim must be greater than zero."
                )

            if fusion_embedding_dim <= 0:
                raise ValueError(
                    "fusion_embedding_dim must be greater than zero."
                )

            if output_dim != 1:
                raise ValueError(
                    "GatedFusionClassifier requires output_dim=1."
                )

            self.fusion_embedding_dim = fusion_embedding_dim

            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            if use_layer_norm:
                self.video_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )

                self.text_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )
            else:
                self.video_projection_normalization = nn.Identity()
                self.text_projection_normalization = nn.Identity()

            self.projection_activation = nn.ReLU()

            self.gate_network = nn.Sequential(
                nn.Linear(
                    fusion_embedding_dim * 2,
                    fusion_embedding_dim,
                ),
                nn.Sigmoid(),
            )

            scoring_layers = [
                nn.Linear(
                    fusion_embedding_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_1)
                )

            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_2)
                )

            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(*scoring_layers)

        def forward(self, video, question_answers):

            if video.ndim != 2:
                raise ValueError(
                    "Expected video tensor shape "
                    "[batch_size, video_embedding_dimension], "
                    f"found {tuple(video.shape)}."
                )

            if question_answers.ndim != 3:
                raise ValueError(
                    "Expected question_answers tensor shape "
                    "[batch_size, num_choices, text_embedding_dimension], "
                    f"found {tuple(question_answers.shape)}."
                )

            batch_size, num_choices, _ = question_answers.shape

            if video.shape[0] != batch_size:
                raise ValueError(
                    "Video and question-answer batch sizes do not match. "
                    f"Video batch={video.shape[0]}, "
                    f"question-answer batch={batch_size}."
                )

            if num_choices != NUM_MULTIPLE_CHOICE_ANSWERS:
                raise ValueError(
                    f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} candidate "
                    f"answers per sample, found {num_choices}."
                )

            video_projected = self.video_projection(video)

            video_projected = (
                self.video_projection_normalization(
                    video_projected
                )
            )

            video_projected = self.projection_activation(
                video_projected
            )

            question_answers_projected = self.text_projection(
                question_answers
            )

            question_answers_projected = (
                self.text_projection_normalization(
                    question_answers_projected
                )
            )

            question_answers_projected = self.projection_activation(
                question_answers_projected
            )

            video_expanded = video_projected.unsqueeze(1).expand(
                -1,
                num_choices,
                -1,
            )

            gate_input = torch.cat(
                [
                    video_expanded,
                    question_answers_projected,
                ],
                dim=-1,
            )

            fusion_gate = self.gate_network(gate_input)

            gated_fusion = (
                fusion_gate * question_answers_projected
                + (1.0 - fusion_gate) * video_expanded
            )

            scores = self.scorer(gated_fusion).squeeze(-1)

            expected_score_shape = (
                batch_size,
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if scores.shape != expected_score_shape:
                raise RuntimeError(
                    "Gated Fusion produced an unexpected score shape. "
                    f"Expected {expected_score_shape}, "
                    f"found {tuple(scores.shape)}."
                )

            return scores

    # ------------------------------------------------------------
    # Initialize model, loss, and optimizer
    # ------------------------------------------------------------

    fusion_model = GatedFusionClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    ).to(device)

    trainable_parameter_count = sum(
        parameter.numel()
        for parameter in fusion_model.parameters()
        if parameter.requires_grad
    )

    if trainable_parameter_count <= 0:
        raise RuntimeError(
            "The Gated Fusion model contains no trainable parameters."
        )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        fusion_model.parameters(),
        lr=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
    )

    # ------------------------------------------------------------
    # Display training configuration
    # ------------------------------------------------------------

    print("Training Gated Fusion classifier.")
    print(f"Device                    : {device}")
    print(f"Training QA samples       : {len(train_qa_samples):,}")
    print(
        f"Validation QA samples     : "
        f"{len(validation_qa_samples):,}"
    )
    print(
        f"Training batches          : "
        f"{len(train_gated_fusion_loader):,}"
    )
    print(
        f"Validation batches        : "
        f"{len(validation_gated_fusion_loader):,}"
    )
    print(
        f"Answer choices per QA     : "
        f"{NUM_MULTIPLE_CHOICE_ANSWERS}"
    )
    print(
        f"Video embedding dimension : "
        f"{video_embedding_dimension}"
    )
    print(
        f"Text embedding dimension  : "
        f"{text_embedding_dimension}"
    )
    print(
        f"Fusion embedding dimension: "
        f"{FUSION_EMBEDDING_DIM}"
    )
    print(
        f"Trainable parameters      : "
        f"{trainable_parameter_count:,}"
    )
    print(f"Batch size                : {FUSION_BATCH_SIZE}")
    print(f"Learning rate             : {FUSION_LEARNING_RATE}")
    print(f"Weight decay              : {FUSION_WEIGHT_DECAY}")
    print(f"Training epochs           : {FUSION_EPOCHS}")

    # ------------------------------------------------------------
    # Train Gated Fusion classifier
    # ------------------------------------------------------------

    fusion_model.train()

    training_history = []

    for epoch in range(FUSION_EPOCHS):

        epoch_losses = []
        epoch_correct = 0
        epoch_record_count = 0

        for batch_index, batch in enumerate(
            train_gated_fusion_loader
        ):

            video_batch = batch["video"].to(device)
            question_answer_batch = batch[
                "question_answers"
            ].to(device)
            label_batch = batch["label"].to(device)

            if video_batch.shape[1] != video_embedding_dimension:
                raise ValueError(
                    "Training video batch dimension mismatch. "
                    f"Expected {video_embedding_dimension}, "
                    f"found {video_batch.shape[1]}."
                )

            expected_question_answer_batch_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                tuple(question_answer_batch.shape)
                != expected_question_answer_batch_shape
            ):
                raise ValueError(
                    "Training question-answer batch shape mismatch. "
                    f"Expected {expected_question_answer_batch_shape}, "
                    f"found {tuple(question_answer_batch.shape)}."
                )

            if label_batch.ndim != 1:
                raise ValueError(
                    "Training labels must be one-dimensional. "
                    f"Found shape {tuple(label_batch.shape)}."
                )

            if torch.any(label_batch < 0) or torch.any(
                label_batch >= NUM_MULTIPLE_CHOICE_ANSWERS
            ):
                raise ValueError(
                    "Training batch contains an invalid answer label."
                )

            optimizer.zero_grad(set_to_none=True)

            scores = fusion_model(
                video_batch,
                question_answer_batch,
            )

            if not torch.isfinite(scores).all():
                raise FloatingPointError(
                    f"Non-finite training scores encountered during "
                    f"epoch {epoch + 1}, batch {batch_index + 1}."
                )

            loss = criterion(scores, label_batch)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite training loss encountered during "
                    f"epoch {epoch + 1}, batch {batch_index + 1}."
                )

            loss.backward()

            non_finite_gradient_parameters = [
                name
                for name, parameter in fusion_model.named_parameters()
                if (
                    parameter.grad is not None
                    and not torch.isfinite(parameter.grad).all()
                )
            ]

            if non_finite_gradient_parameters:
                raise FloatingPointError(
                    "Non-finite gradients detected in Gated Fusion "
                    "parameters: "
                    + ", ".join(non_finite_gradient_parameters)
                )

            optimizer.step()

            batch_record_count = label_batch.shape[0]
            batch_predictions = torch.argmax(scores, dim=1)

            epoch_losses.append(float(loss.item()))
            epoch_correct += int(
                (batch_predictions == label_batch).sum().item()
            )
            epoch_record_count += int(batch_record_count)

        if not epoch_losses:
            raise RuntimeError(
                f"No training batches were processed during epoch "
                f"{epoch + 1}."
            )

        if epoch_record_count != len(train_gated_fusion_dataset):
            raise RuntimeError(
                f"Epoch {epoch + 1} processed "
                f"{epoch_record_count:,} training records; expected "
                f"{len(train_gated_fusion_dataset):,}."
            )

        mean_epoch_loss = float(np.mean(epoch_losses))

        if not np.isfinite(mean_epoch_loss):
            raise FloatingPointError(
                f"Epoch {epoch + 1} produced a non-finite mean loss."
            )

        epoch_accuracy = float(
            epoch_correct / epoch_record_count
        )

        training_history.append(
            {
                "epoch": epoch + 1,
                "loss": mean_epoch_loss,
                "training_accuracy": epoch_accuracy,
                "records": epoch_record_count,
                "batches": len(epoch_losses),
            }
        )

        print(
            f"Epoch {epoch + 1:02d}/{FUSION_EPOCHS:02d} "
            f"- loss: {mean_epoch_loss:.4f} "
            f"- training accuracy: {epoch_accuracy:.3f}"
        )

    fusion_training_history_df = pd.DataFrame(
        training_history
    )

    # ------------------------------------------------------------
    # Validate training history
    # ------------------------------------------------------------

    required_training_history_columns = [
        "epoch",
        "loss",
        "training_accuracy",
        "records",
        "batches",
    ]

    missing_training_history_columns = [
        column
        for column in required_training_history_columns
        if column not in fusion_training_history_df.columns
    ]

    if missing_training_history_columns:
        raise RuntimeError(
            "Gated Fusion training history is missing columns: "
            + ", ".join(missing_training_history_columns)
        )

    if len(fusion_training_history_df) != FUSION_EPOCHS:
        raise RuntimeError(
            "Gated Fusion training history record count mismatch. "
            f"Expected {FUSION_EPOCHS}, found "
            f"{len(fusion_training_history_df)}."
        )

    expected_epoch_values = list(
        range(1, FUSION_EPOCHS + 1)
    )

    actual_epoch_values = (
        fusion_training_history_df["epoch"]
        .astype(int)
        .tolist()
    )

    if actual_epoch_values != expected_epoch_values:
        raise RuntimeError(
            "Gated Fusion training history contains invalid epoch "
            f"values: {actual_epoch_values}"
        )

    if (
        fusion_training_history_df["loss"]
        .isna()
        .any()
    ):
        raise RuntimeError(
            "Gated Fusion training history contains missing loss values."
        )

    if not np.isfinite(
        fusion_training_history_df["loss"].to_numpy(
            dtype=np.float64
        )
    ).all():
        raise FloatingPointError(
            "Gated Fusion training history contains non-finite losses."
        )

    invalid_training_accuracy_rows = (
        fusion_training_history_df[
            ~fusion_training_history_df[
                "training_accuracy"
            ].between(0.0, 1.0)
        ]
    )

    if not invalid_training_accuracy_rows.empty:
        raise RuntimeError(
            "Gated Fusion training history contains invalid "
            "training-accuracy values."
        )

    # ------------------------------------------------------------
    # Generate validation scores
    # ------------------------------------------------------------

    fusion_model.eval()

    validation_score_batches = []
    validation_record_count = 0

    with torch.no_grad():

        for batch_index, batch in enumerate(
            validation_gated_fusion_loader
        ):

            video_batch = batch["video"].to(device)
            question_answer_batch = batch[
                "question_answers"
            ].to(device)

            if video_batch.shape[1] != video_embedding_dimension:
                raise ValueError(
                    "Validation video batch dimension mismatch. "
                    f"Expected {video_embedding_dimension}, "
                    f"found {video_batch.shape[1]}."
                )

            expected_question_answer_batch_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                tuple(question_answer_batch.shape)
                != expected_question_answer_batch_shape
            ):
                raise ValueError(
                    "Validation question-answer batch shape mismatch. "
                    f"Expected {expected_question_answer_batch_shape}, "
                    f"found {tuple(question_answer_batch.shape)}."
                )

            scores = fusion_model(
                video_batch,
                question_answer_batch,
            )

            expected_score_batch_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if tuple(scores.shape) != expected_score_batch_shape:
                raise RuntimeError(
                    "Validation score batch shape mismatch. "
                    f"Expected {expected_score_batch_shape}, "
                    f"found {tuple(scores.shape)}."
                )

            if not torch.isfinite(scores).all():
                raise FloatingPointError(
                    f"Non-finite validation scores encountered in "
                    f"batch {batch_index + 1}."
                )

            score_batch_array = (
                scores.detach()
                .cpu()
                .numpy()
                .astype(np.float32, copy=False)
            )

            validation_score_batches.append(
                score_batch_array
            )

            validation_record_count += int(
                score_batch_array.shape[0]
            )

    if not validation_score_batches:
        raise RuntimeError(
            "Gated Fusion inference produced no validation scores."
        )

    if (
        validation_record_count
        != len(validation_gated_fusion_dataset)
    ):
        raise RuntimeError(
            "Gated Fusion validation inference record-count mismatch. "
            f"Expected {len(validation_gated_fusion_dataset):,}, "
            f"processed {validation_record_count:,}."
        )

    all_scores = np.vstack(validation_score_batches)

    # ------------------------------------------------------------
    # Validate shared score output
    # ------------------------------------------------------------

    expected_all_scores_shape = (
        len(validation_qa_samples),
        NUM_MULTIPLE_CHOICE_ANSWERS,
    )

    if all_scores.shape != expected_all_scores_shape:
        raise RuntimeError(
            "Gated Fusion score matrix shape mismatch. "
            f"Expected {expected_all_scores_shape}, "
            f"found {all_scores.shape}."
        )

    if not np.issubdtype(all_scores.dtype, np.number):
        raise TypeError(
            "Gated Fusion all_scores must contain numeric values."
        )

    if not np.isfinite(all_scores).all():
        raise FloatingPointError(
            "Gated Fusion all_scores contains non-finite values."
        )

    if np.any(np.ptp(all_scores, axis=1) == 0):
        identical_score_count = int(
            np.sum(np.ptp(all_scores, axis=1) == 0)
        )

        print(
            "Warning: Gated Fusion produced identical candidate scores "
            f"for {identical_score_count:,} validation QA records."
        )

    # ------------------------------------------------------------
    # Populate shared method-output contract
    # ------------------------------------------------------------

    training_epochs = int(FUSION_EPOCHS)

    final_training_loss = float(
        fusion_training_history_df["loss"].iloc[-1]
    )

    fusion_embedding_dimension = int(
        FUSION_EMBEDDING_DIM
    )

    if training_epochs <= 0:
        raise RuntimeError(
            "training_epochs was not populated correctly."
        )

    if not np.isfinite(final_training_loss):
        raise FloatingPointError(
            "final_training_loss is not finite."
        )

    if fusion_embedding_dimension <= 0:
        raise RuntimeError(
            "fusion_embedding_dimension was not populated correctly."
        )

    # ------------------------------------------------------------
    # Display completion summary
    # ------------------------------------------------------------

    validation_predicted_choices = np.argmax(
        all_scores,
        axis=1,
    )

    validation_ground_truth_choices = np.array(
        [
            int(sample["label"])
            for sample in validation_qa_samples
        ],
        dtype=np.int64,
    )

    validation_choice_accuracy = float(
        np.mean(
            validation_predicted_choices
            == validation_ground_truth_choices
        )
    )

    print("Gated Fusion training and validation scoring completed.")
    print(f"Device                    : {device}")
    print(
        f"Training QA samples       : "
        f"{len(train_qa_samples):,}"
    )
    print(
        f"Validation QA samples     : "
        f"{len(validation_qa_samples):,}"
    )
    print(
        f"Validation score rows     : "
        f"{all_scores.shape[0]:,}"
    )
    print(
        f"Scores per QA record      : "
        f"{all_scores.shape[1]}"
    )
    print(
        f"Video embedding dimension : "
        f"{video_embedding_dimension}"
    )
    print(
        f"Text embedding dimension  : "
        f"{text_embedding_dimension}"
    )
    print(
        f"Fusion embedding dimension: "
        f"{fusion_embedding_dimension}"
    )
    print(f"Training epochs           : {training_epochs}")
    print(
        f"Final training loss       : "
        f"{final_training_loss:.4f}"
    )
    print(
        f"Validation choice accuracy: "
        f"{validation_choice_accuracy:.3f}"
    )

else:
    print(
        "Skipping Gated Fusion classifier because "
        f"REPRESENTATION_VIDEOQA_METHOD="
        f"'{REPRESENTATION_VIDEOQA_METHOD}'."
    )



### 🔷 Step 13 — Bilinear Fusion



In [ ]:
# ============================================================
# Step 13: Run Bilinear Fusion Classifier
# ============================================================

print("Evaluating Bilinear Fusion classifier method selection.")

if REPRESENTATION_VIDEOQA_METHOD == "bilinear_fusion_classifier":

    # ------------------------------------------------------------
    # Define computation device when not already available
    # ------------------------------------------------------------

    if "torch" not in globals():
        raise NameError(
            "PyTorch must be imported before running Step 13."
        )

    if "device" not in globals():
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    if not isinstance(device, torch.device):
        raise TypeError(
            "device must be a torch.device object. "
            f"Found {type(device).__name__}."
        )

    print(f"Computation device selected: {device}")

    # ------------------------------------------------------------
    # Verify required inputs
    # ------------------------------------------------------------

    required_bilinear_fusion_objects = [
        "train_qa_samples",
        "validation_qa_samples",
        "device",
        "text_embedding_dimension",
        "video_embedding_dimension",
        "NUM_MULTIPLE_CHOICE_ANSWERS",
        "FUSION_EMBEDDING_DIM",
        "FUSION_HIDDEN_DIM_1",
        "FUSION_HIDDEN_DIM_2",
        "FUSION_OUTPUT_DIM",
        "FUSION_DROPOUT",
        "FUSION_BATCH_SIZE",
        "FUSION_LEARNING_RATE",
        "FUSION_WEIGHT_DECAY",
        "FUSION_EPOCHS",
        "FUSION_RANDOM_SEED",
        "USE_LAYER_NORMALIZATION",
    ]

    missing_bilinear_fusion_objects = [
        name
        for name in required_bilinear_fusion_objects
        if name not in globals()
    ]

    if missing_bilinear_fusion_objects:
        raise NameError(
            "Missing required objects for Bilinear Fusion classification: "
            + ", ".join(missing_bilinear_fusion_objects)
        )

    if not isinstance(train_qa_samples, list) or not train_qa_samples:
        raise RuntimeError(
            "train_qa_samples must be a non-empty list."
        )

    if (
        not isinstance(validation_qa_samples, list)
        or not validation_qa_samples
    ):
        raise RuntimeError(
            "validation_qa_samples must be a non-empty list."
        )

    if text_embedding_dimension <= 0:
        raise ValueError(
            "text_embedding_dimension must be greater than zero."
        )

    if video_embedding_dimension <= 0:
        raise ValueError(
            "video_embedding_dimension must be greater than zero."
        )

    if NUM_MULTIPLE_CHOICE_ANSWERS <= 1:
        raise ValueError(
            "NUM_MULTIPLE_CHOICE_ANSWERS must be greater than one."
        )

    if FUSION_EMBEDDING_DIM <= 0:
        raise ValueError(
            "FUSION_EMBEDDING_DIM must be greater than zero."
        )

    if FUSION_HIDDEN_DIM_1 <= 0:
        raise ValueError(
            "FUSION_HIDDEN_DIM_1 must be greater than zero."
        )

    if FUSION_HIDDEN_DIM_2 <= 0:
        raise ValueError(
            "FUSION_HIDDEN_DIM_2 must be greater than zero."
        )

    if FUSION_OUTPUT_DIM != 1:
        raise ValueError(
            "Bilinear Fusion requires FUSION_OUTPUT_DIM=1 so that "
            "each candidate answer receives one scalar score. "
            f"Found FUSION_OUTPUT_DIM={FUSION_OUTPUT_DIM}."
        )

    if not 0.0 <= FUSION_DROPOUT < 1.0:
        raise ValueError(
            "FUSION_DROPOUT must be in the range [0.0, 1.0). "
            f"Found {FUSION_DROPOUT}."
        )

    if FUSION_BATCH_SIZE <= 0:
        raise ValueError(
            "FUSION_BATCH_SIZE must be greater than zero."
        )

    if FUSION_LEARNING_RATE <= 0:
        raise ValueError(
            "FUSION_LEARNING_RATE must be greater than zero."
        )

    if FUSION_WEIGHT_DECAY < 0:
        raise ValueError(
            "FUSION_WEIGHT_DECAY cannot be negative."
        )

    if FUSION_EPOCHS <= 0:
        raise ValueError(
            "FUSION_EPOCHS must be greater than zero."
        )

    # ------------------------------------------------------------
    # Validate grouped QA sample structure
    # ------------------------------------------------------------

    def validate_bilinear_fusion_samples(
        samples,
        sample_group_name,
    ):

        required_sample_fields = {
            "qa_record_id",
            "video_embedding",
            "question_answer_embeddings",
            "label",
        }

        for sample_index, sample in enumerate(samples):

            if not isinstance(sample, dict):
                raise TypeError(
                    f"{sample_group_name} sample {sample_index} must "
                    "be a dictionary."
                )

            missing_sample_fields = sorted(
                required_sample_fields - set(sample.keys())
            )

            if missing_sample_fields:
                raise KeyError(
                    f"{sample_group_name} sample {sample_index} is "
                    "missing required fields: "
                    + ", ".join(missing_sample_fields)
                )

            video_embedding = np.asarray(
                sample["video_embedding"],
                dtype=np.float32,
            )

            question_answer_embeddings = np.asarray(
                sample["question_answer_embeddings"],
                dtype=np.float32,
            )

            if video_embedding.ndim != 1:
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} video "
                    "embedding must be one-dimensional. "
                    f"Found shape {video_embedding.shape}."
                )

            if (
                video_embedding.shape[0]
                != video_embedding_dimension
            ):
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} video "
                    "embedding dimension mismatch. Expected "
                    f"{video_embedding_dimension}, found "
                    f"{video_embedding.shape[0]}."
                )

            expected_question_answer_shape = (
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                question_answer_embeddings.shape
                != expected_question_answer_shape
            ):
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} "
                    "question-answer embedding shape mismatch. Expected "
                    f"{expected_question_answer_shape}, found "
                    f"{question_answer_embeddings.shape}."
                )

            label = sample["label"]

            if isinstance(label, bool):
                raise TypeError(
                    f"{sample_group_name} sample {sample_index} label "
                    "cannot be boolean."
                )

            try:
                label = int(label)
            except (TypeError, ValueError) as exc:
                raise TypeError(
                    f"{sample_group_name} sample {sample_index} has a "
                    f"non-integer label: {label}"
                ) from exc

            if label not in range(NUM_MULTIPLE_CHOICE_ANSWERS):
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} has "
                    f"invalid label {label}."
                )

            if not np.isfinite(video_embedding).all():
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} contains "
                    "non-finite video embedding values."
                )

            if not np.isfinite(question_answer_embeddings).all():
                raise ValueError(
                    f"{sample_group_name} sample {sample_index} contains "
                    "non-finite question-answer embedding values."
                )

    validate_bilinear_fusion_samples(
        samples=train_qa_samples,
        sample_group_name="Training",
    )

    validate_bilinear_fusion_samples(
        samples=validation_qa_samples,
        sample_group_name="Validation",
    )

    # ------------------------------------------------------------
    # Configure reproducibility
    # ------------------------------------------------------------

    np.random.seed(FUSION_RANDOM_SEED)
    torch.manual_seed(FUSION_RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(FUSION_RANDOM_SEED)
        torch.cuda.manual_seed_all(FUSION_RANDOM_SEED)

    data_loader_generator = torch.Generator()
    data_loader_generator.manual_seed(FUSION_RANDOM_SEED)

    # ------------------------------------------------------------
    # Define grouped QA dataset
    # ------------------------------------------------------------

    class BilinearFusionQADataset(Dataset):

        def __init__(self, samples):

            if not samples:
                raise RuntimeError(
                    "BilinearFusionQADataset received no samples."
                )

            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):

            sample = self.samples[idx]

            return {
                "video": torch.tensor(
                    sample["video_embedding"],
                    dtype=torch.float32,
                ),
                "question_answers": torch.tensor(
                    sample["question_answer_embeddings"],
                    dtype=torch.float32,
                ),
                "label": torch.tensor(
                    int(sample["label"]),
                    dtype=torch.long,
                ),
            }

    # ------------------------------------------------------------
    # Build training and validation data loaders
    # ------------------------------------------------------------

    train_bilinear_fusion_dataset = BilinearFusionQADataset(
        train_qa_samples
    )

    validation_bilinear_fusion_dataset = (
        BilinearFusionQADataset(
            validation_qa_samples
        )
    )

    train_bilinear_fusion_loader = DataLoader(
        train_bilinear_fusion_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=True,
        generator=data_loader_generator,
        drop_last=False,
    )

    validation_bilinear_fusion_loader = DataLoader(
        validation_bilinear_fusion_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )

    if len(train_bilinear_fusion_loader) == 0:
        raise RuntimeError(
            "The Bilinear Fusion training data loader contains no "
            "batches."
        )

    if len(validation_bilinear_fusion_loader) == 0:
        raise RuntimeError(
            "The Bilinear Fusion validation data loader contains no "
            "batches."
        )

    # ------------------------------------------------------------
    # Define Bilinear Fusion classifier
    # ------------------------------------------------------------

    class BilinearFusionClassifier(nn.Module):

        def __init__(
            self,
            video_input_dim,
            text_input_dim,
            fusion_embedding_dim,
            hidden_dim_1,
            hidden_dim_2,
            output_dim,
            dropout,
            use_layer_norm=True,
        ):
            super().__init__()

            if video_input_dim <= 0:
                raise ValueError(
                    "video_input_dim must be greater than zero."
                )

            if text_input_dim <= 0:
                raise ValueError(
                    "text_input_dim must be greater than zero."
                )

            if fusion_embedding_dim <= 0:
                raise ValueError(
                    "fusion_embedding_dim must be greater than zero."
                )

            if hidden_dim_1 <= 0 or hidden_dim_2 <= 0:
                raise ValueError(
                    "Bilinear Fusion hidden dimensions must be greater "
                    "than zero."
                )

            if output_dim != 1:
                raise ValueError(
                    "BilinearFusionClassifier requires output_dim=1."
                )

            self.video_input_dim = video_input_dim
            self.text_input_dim = text_input_dim
            self.fusion_embedding_dim = fusion_embedding_dim

            self.video_projection = nn.Linear(
                video_input_dim,
                fusion_embedding_dim,
            )

            self.text_projection = nn.Linear(
                text_input_dim,
                fusion_embedding_dim,
            )

            if use_layer_norm:
                self.video_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )

                self.text_projection_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )

                self.bilinear_normalization = nn.LayerNorm(
                    fusion_embedding_dim
                )
            else:
                self.video_projection_normalization = nn.Identity()
                self.text_projection_normalization = nn.Identity()
                self.bilinear_normalization = nn.Identity()

            self.projection_activation = nn.Tanh()

            scoring_layers = [
                nn.Linear(
                    fusion_embedding_dim,
                    hidden_dim_1,
                ),
                nn.ReLU(),
            ]

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_1)
                )

            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_1,
                        hidden_dim_2,
                    ),
                    nn.ReLU(),
                ]
            )

            if use_layer_norm:
                scoring_layers.append(
                    nn.LayerNorm(hidden_dim_2)
                )

            scoring_layers.extend(
                [
                    nn.Dropout(dropout),
                    nn.Linear(
                        hidden_dim_2,
                        output_dim,
                    ),
                ]
            )

            self.scorer = nn.Sequential(*scoring_layers)

        def forward(self, video, question_answers):

            if video.ndim != 2:
                raise ValueError(
                    "Expected video tensor shape "
                    "[batch_size, video_embedding_dimension], "
                    f"found {tuple(video.shape)}."
                )

            if question_answers.ndim != 3:
                raise ValueError(
                    "Expected question_answers tensor shape "
                    "[batch_size, num_choices, "
                    "text_embedding_dimension], "
                    f"found {tuple(question_answers.shape)}."
                )

            batch_size, num_choices, text_dimension = (
                question_answers.shape
            )

            if video.shape[0] != batch_size:
                raise ValueError(
                    "Video and question-answer batch sizes do not "
                    "match. "
                    f"Video batch={video.shape[0]}, "
                    f"question-answer batch={batch_size}."
                )

            if video.shape[1] != self.video_input_dim:
                raise ValueError(
                    "Video input dimension mismatch. Expected "
                    f"{self.video_input_dim}, found {video.shape[1]}."
                )

            if text_dimension != self.text_input_dim:
                raise ValueError(
                    "Question-answer input dimension mismatch. Expected "
                    f"{self.text_input_dim}, found {text_dimension}."
                )

            if num_choices != NUM_MULTIPLE_CHOICE_ANSWERS:
                raise ValueError(
                    f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} candidate "
                    f"answers per sample, found {num_choices}."
                )

            video_projected = self.video_projection(video)

            video_projected = (
                self.video_projection_normalization(
                    video_projected
                )
            )

            video_projected = self.projection_activation(
                video_projected
            )

            question_answers_projected = self.text_projection(
                question_answers
            )

            question_answers_projected = (
                self.text_projection_normalization(
                    question_answers_projected
                )
            )

            question_answers_projected = self.projection_activation(
                question_answers_projected
            )

            video_expanded = video_projected.unsqueeze(1).expand(
                -1,
                num_choices,
                -1,
            )

            # Factorized bilinear interaction:
            # projected video and text features are multiplied
            # elementwise in the shared fusion space.
            bilinear_fusion = (
                video_expanded
                * question_answers_projected
            )

            bilinear_fusion = self.bilinear_normalization(
                bilinear_fusion
            )

            scores = self.scorer(
                bilinear_fusion
            ).squeeze(-1)

            expected_score_shape = (
                batch_size,
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if tuple(scores.shape) != expected_score_shape:
                raise RuntimeError(
                    "Bilinear Fusion produced an unexpected score "
                    "shape. "
                    f"Expected {expected_score_shape}, "
                    f"found {tuple(scores.shape)}."
                )

            return scores

    # ------------------------------------------------------------
    # Initialize model, loss, and optimizer
    # ------------------------------------------------------------

    fusion_model = BilinearFusionClassifier(
        video_input_dim=video_embedding_dimension,
        text_input_dim=text_embedding_dimension,
        fusion_embedding_dim=FUSION_EMBEDDING_DIM,
        hidden_dim_1=FUSION_HIDDEN_DIM_1,
        hidden_dim_2=FUSION_HIDDEN_DIM_2,
        output_dim=FUSION_OUTPUT_DIM,
        dropout=FUSION_DROPOUT,
        use_layer_norm=USE_LAYER_NORMALIZATION,
    ).to(device)

    trainable_parameter_count = sum(
        parameter.numel()
        for parameter in fusion_model.parameters()
        if parameter.requires_grad
    )

    if trainable_parameter_count <= 0:
        raise RuntimeError(
            "The Bilinear Fusion model contains no trainable "
            "parameters."
        )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        fusion_model.parameters(),
        lr=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY,
    )

    # ------------------------------------------------------------
    # Display training configuration
    # ------------------------------------------------------------

    print("Training Bilinear Fusion classifier.")
    print(f"Device                    : {device}")
    print(
        f"Training QA samples       : "
        f"{len(train_qa_samples):,}"
    )
    print(
        f"Validation QA samples     : "
        f"{len(validation_qa_samples):,}"
    )
    print(
        f"Training batches          : "
        f"{len(train_bilinear_fusion_loader):,}"
    )
    print(
        f"Validation batches        : "
        f"{len(validation_bilinear_fusion_loader):,}"
    )
    print(
        f"Answer choices per QA     : "
        f"{NUM_MULTIPLE_CHOICE_ANSWERS}"
    )
    print(
        f"Video embedding dimension : "
        f"{video_embedding_dimension}"
    )
    print(
        f"Text embedding dimension  : "
        f"{text_embedding_dimension}"
    )
    print(
        f"Fusion embedding dimension: "
        f"{FUSION_EMBEDDING_DIM}"
    )
    print(
        f"Trainable parameters      : "
        f"{trainable_parameter_count:,}"
    )
    print(f"Batch size                : {FUSION_BATCH_SIZE}")
    print(f"Learning rate             : {FUSION_LEARNING_RATE}")
    print(f"Weight decay              : {FUSION_WEIGHT_DECAY}")
    print(f"Dropout                    : {FUSION_DROPOUT}")
    print(f"Training epochs           : {FUSION_EPOCHS}")

    # ------------------------------------------------------------
    # Train Bilinear Fusion classifier
    # ------------------------------------------------------------

    fusion_model.train()

    training_history = []

    for epoch in range(FUSION_EPOCHS):

        epoch_losses = []
        epoch_correct = 0
        epoch_record_count = 0

        for batch_index, batch in enumerate(
            train_bilinear_fusion_loader
        ):

            video_batch = batch["video"].to(device)

            question_answer_batch = batch[
                "question_answers"
            ].to(device)

            label_batch = batch["label"].to(device)

            if video_batch.ndim != 2:
                raise ValueError(
                    "Training video batch must be two-dimensional. "
                    f"Found shape {tuple(video_batch.shape)}."
                )

            if (
                video_batch.shape[1]
                != video_embedding_dimension
            ):
                raise ValueError(
                    "Training video batch dimension mismatch. "
                    f"Expected {video_embedding_dimension}, "
                    f"found {video_batch.shape[1]}."
                )

            expected_question_answer_batch_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                tuple(question_answer_batch.shape)
                != expected_question_answer_batch_shape
            ):
                raise ValueError(
                    "Training question-answer batch shape mismatch. "
                    f"Expected "
                    f"{expected_question_answer_batch_shape}, "
                    f"found "
                    f"{tuple(question_answer_batch.shape)}."
                )

            if label_batch.ndim != 1:
                raise ValueError(
                    "Training labels must be one-dimensional. "
                    f"Found shape {tuple(label_batch.shape)}."
                )

            if torch.any(label_batch < 0) or torch.any(
                label_batch >= NUM_MULTIPLE_CHOICE_ANSWERS
            ):
                raise ValueError(
                    "Training batch contains an invalid answer label."
                )

            optimizer.zero_grad(set_to_none=True)

            scores = fusion_model(
                video_batch,
                question_answer_batch,
            )

            if not torch.isfinite(scores).all():
                raise FloatingPointError(
                    "Non-finite training scores encountered during "
                    f"epoch {epoch + 1}, batch {batch_index + 1}."
                )

            loss = criterion(
                scores,
                label_batch,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Non-finite training loss encountered during "
                    f"epoch {epoch + 1}, batch {batch_index + 1}."
                )

            loss.backward()

            non_finite_gradient_parameters = [
                name
                for name, parameter
                in fusion_model.named_parameters()
                if (
                    parameter.grad is not None
                    and not torch.isfinite(
                        parameter.grad
                    ).all()
                )
            ]

            if non_finite_gradient_parameters:
                raise FloatingPointError(
                    "Non-finite gradients detected in Bilinear Fusion "
                    "parameters: "
                    + ", ".join(
                        non_finite_gradient_parameters
                    )
                )

            optimizer.step()

            batch_predictions = torch.argmax(
                scores,
                dim=1,
            )

            batch_record_count = int(
                label_batch.shape[0]
            )

            epoch_losses.append(
                float(loss.item())
            )

            epoch_correct += int(
                (
                    batch_predictions
                    == label_batch
                ).sum().item()
            )

            epoch_record_count += batch_record_count

        if not epoch_losses:
            raise RuntimeError(
                f"No training batches were processed during epoch "
                f"{epoch + 1}."
            )

        if (
            epoch_record_count
            != len(train_bilinear_fusion_dataset)
        ):
            raise RuntimeError(
                f"Epoch {epoch + 1} processed "
                f"{epoch_record_count:,} training records; expected "
                f"{len(train_bilinear_fusion_dataset):,}."
            )

        mean_epoch_loss = float(
            np.mean(epoch_losses)
        )

        if not np.isfinite(mean_epoch_loss):
            raise FloatingPointError(
                f"Epoch {epoch + 1} produced a non-finite mean loss."
            )

        epoch_accuracy = float(
            epoch_correct / epoch_record_count
        )

        training_history.append(
            {
                "epoch": epoch + 1,
                "loss": mean_epoch_loss,
                "training_accuracy": epoch_accuracy,
                "records": epoch_record_count,
                "batches": len(epoch_losses),
            }
        )

        print(
            f"Epoch {epoch + 1:02d}/{FUSION_EPOCHS:02d} "
            f"- loss: {mean_epoch_loss:.4f} "
            f"- training accuracy: {epoch_accuracy:.3f}"
        )

    fusion_training_history_df = pd.DataFrame(
        training_history
    )

    # ------------------------------------------------------------
    # Validate training history
    # ------------------------------------------------------------

    required_training_history_columns = [
        "epoch",
        "loss",
        "training_accuracy",
        "records",
        "batches",
    ]

    missing_training_history_columns = [
        column
        for column in required_training_history_columns
        if column not in fusion_training_history_df.columns
    ]

    if missing_training_history_columns:
        raise RuntimeError(
            "Bilinear Fusion training history is missing columns: "
            + ", ".join(missing_training_history_columns)
        )

    if len(fusion_training_history_df) != FUSION_EPOCHS:
        raise RuntimeError(
            "Bilinear Fusion training history record-count mismatch. "
            f"Expected {FUSION_EPOCHS}, found "
            f"{len(fusion_training_history_df)}."
        )

    expected_epoch_values = list(
        range(1, FUSION_EPOCHS + 1)
    )

    actual_epoch_values = (
        fusion_training_history_df["epoch"]
        .astype(int)
        .tolist()
    )

    if actual_epoch_values != expected_epoch_values:
        raise RuntimeError(
            "Bilinear Fusion training history contains invalid epoch "
            f"values: {actual_epoch_values}"
        )

    training_loss_values = (
        fusion_training_history_df["loss"]
        .to_numpy(dtype=np.float64)
    )

    if not np.isfinite(training_loss_values).all():
        raise FloatingPointError(
            "Bilinear Fusion training history contains non-finite "
            "losses."
        )

    invalid_training_accuracy_rows = (
        fusion_training_history_df[
            ~fusion_training_history_df[
                "training_accuracy"
            ].between(0.0, 1.0)
        ]
    )

    if not invalid_training_accuracy_rows.empty:
        raise RuntimeError(
            "Bilinear Fusion training history contains invalid "
            "training-accuracy values."
        )

    # ------------------------------------------------------------
    # Generate validation scores
    # ------------------------------------------------------------

    fusion_model.eval()

    validation_score_batches = []
    validation_record_count = 0

    with torch.no_grad():

        for batch_index, batch in enumerate(
            validation_bilinear_fusion_loader
        ):

            video_batch = batch["video"].to(device)

            question_answer_batch = batch[
                "question_answers"
            ].to(device)

            if video_batch.ndim != 2:
                raise ValueError(
                    "Validation video batch must be "
                    "two-dimensional. "
                    f"Found shape {tuple(video_batch.shape)}."
                )

            if (
                video_batch.shape[1]
                != video_embedding_dimension
            ):
                raise ValueError(
                    "Validation video batch dimension mismatch. "
                    f"Expected {video_embedding_dimension}, "
                    f"found {video_batch.shape[1]}."
                )

            expected_question_answer_batch_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
                text_embedding_dimension,
            )

            if (
                tuple(question_answer_batch.shape)
                != expected_question_answer_batch_shape
            ):
                raise ValueError(
                    "Validation question-answer batch shape mismatch. "
                    f"Expected "
                    f"{expected_question_answer_batch_shape}, "
                    f"found "
                    f"{tuple(question_answer_batch.shape)}."
                )

            scores = fusion_model(
                video_batch,
                question_answer_batch,
            )

            expected_score_batch_shape = (
                video_batch.shape[0],
                NUM_MULTIPLE_CHOICE_ANSWERS,
            )

            if tuple(scores.shape) != expected_score_batch_shape:
                raise RuntimeError(
                    "Validation score batch shape mismatch. "
                    f"Expected {expected_score_batch_shape}, "
                    f"found {tuple(scores.shape)}."
                )

            if not torch.isfinite(scores).all():
                raise FloatingPointError(
                    "Non-finite validation scores encountered in "
                    f"batch {batch_index + 1}."
                )

            score_batch_array = (
                scores.detach()
                .cpu()
                .numpy()
                .astype(np.float32, copy=False)
            )

            validation_score_batches.append(
                score_batch_array
            )

            validation_record_count += int(
                score_batch_array.shape[0]
            )

    if not validation_score_batches:
        raise RuntimeError(
            "Bilinear Fusion inference produced no validation scores."
        )

    if (
        validation_record_count
        != len(validation_bilinear_fusion_dataset)
    ):
        raise RuntimeError(
            "Bilinear Fusion validation inference record-count "
            "mismatch. "
            f"Expected "
            f"{len(validation_bilinear_fusion_dataset):,}, "
            f"processed {validation_record_count:,}."
        )

    all_scores = np.vstack(
        validation_score_batches
    )

    # ------------------------------------------------------------
    # Validate shared score output
    # ------------------------------------------------------------

    expected_all_scores_shape = (
        len(validation_qa_samples),
        NUM_MULTIPLE_CHOICE_ANSWERS,
    )

    if all_scores.shape != expected_all_scores_shape:
        raise RuntimeError(
            "Bilinear Fusion score matrix shape mismatch. "
            f"Expected {expected_all_scores_shape}, "
            f"found {all_scores.shape}."
        )

    if not np.issubdtype(all_scores.dtype, np.number):
        raise TypeError(
            "Bilinear Fusion all_scores must contain numeric values."
        )

    if not np.isfinite(all_scores).all():
        raise FloatingPointError(
            "Bilinear Fusion all_scores contains non-finite values."
        )

    identical_score_mask = (
        np.ptp(all_scores, axis=1) == 0
    )

    if np.any(identical_score_mask):

        identical_score_count = int(
            np.sum(identical_score_mask)
        )

        print(
            "Warning: Bilinear Fusion produced identical candidate "
            f"scores for {identical_score_count:,} validation QA "
            "records."
        )

    # ------------------------------------------------------------
    # Populate shared method-output contract
    # ------------------------------------------------------------

    training_epochs = int(
        FUSION_EPOCHS
    )

    final_training_loss = float(
        fusion_training_history_df[
            "loss"
        ].iloc[-1]
    )

    fusion_embedding_dimension = int(
        FUSION_EMBEDDING_DIM
    )

    if training_epochs <= 0:
        raise RuntimeError(
            "training_epochs was not populated correctly."
        )

    if not np.isfinite(final_training_loss):
        raise FloatingPointError(
            "final_training_loss is not finite."
        )

    if fusion_embedding_dimension <= 0:
        raise RuntimeError(
            "fusion_embedding_dimension was not populated correctly."
        )

    # ------------------------------------------------------------
    # Display completion summary
    # ------------------------------------------------------------

    validation_predicted_choices = np.argmax(
        all_scores,
        axis=1,
    )

    validation_ground_truth_choices = np.array(
        [
            int(sample["label"])
            for sample in validation_qa_samples
        ],
        dtype=np.int64,
    )

    validation_choice_accuracy = float(
        np.mean(
            validation_predicted_choices
            == validation_ground_truth_choices
        )
    )

    print(
        "Bilinear Fusion training and validation scoring completed."
    )
    print(f"Device                    : {device}")
    print(
        f"Training QA samples       : "
        f"{len(train_qa_samples):,}"
    )
    print(
        f"Validation QA samples     : "
        f"{len(validation_qa_samples):,}"
    )
    print(
        f"Validation score rows     : "
        f"{all_scores.shape[0]:,}"
    )
    print(
        f"Scores per QA record      : "
        f"{all_scores.shape[1]}"
    )
    print(
        f"Video embedding dimension : "
        f"{video_embedding_dimension}"
    )
    print(
        f"Text embedding dimension  : "
        f"{text_embedding_dimension}"
    )
    print(
        f"Fusion embedding dimension: "
        f"{fusion_embedding_dimension}"
    )
    print(f"Training epochs           : {training_epochs}")
    print(
        f"Final training loss       : "
        f"{final_training_loss:.4f}"
    )
    print(
        f"Validation choice accuracy: "
        f"{validation_choice_accuracy:.3f}"
    )

else:
    print(
        "Skipping Bilinear Fusion classifier because "
        f"REPRESENTATION_VIDEOQA_METHOD="
        f"'{REPRESENTATION_VIDEOQA_METHOD}'."
    )



### 🔷 Step 14 — Build and Validate Prediction Datasets

* Verify that the generated prediction dataset contains the expected number of QA prediction records.
* Confirm that one prediction exists for every evaluation question and that all candidate-answer rows were processed.
* Validate required prediction fields, duplicate identifiers, and missing values.
* Verify that all ground-truth and predicted answer indices are valid multiple-choice labels.
* Confirm that prediction correctness values are valid Boolean indicators.
* Display a validation summary confirming the integrity of the representation-based prediction dataset.

In [ ]:
# ============================================================
# Step 14: Build and Validate Prediction Datasets
# ============================================================

import numpy as np
import pandas as pd

print("Building and validating representation-based prediction datasets...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step14_objects = [
    "all_scores",
    "validation_qa_samples",
    "validation_qa_input_df",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
    "video_id_column",
    "question_column",
    "evaluation_split",
]

missing_step14_objects = [
    name
    for name in required_step14_objects
    if name not in globals()
]

if missing_step14_objects:
    raise NameError(
        "Missing required Step 14 objects: "
        + ", ".join(missing_step14_objects)
    )

# ------------------------------------------------------------
# Validate selected-method outputs
# ------------------------------------------------------------

if all_scores is None:
    raise RuntimeError(
        "No prediction scores were generated. "
        "Verify that the selected prediction-method step ran successfully."
    )

all_scores = np.asarray(
    all_scores,
    dtype=np.float32,
)

expected_score_shape = (
    len(validation_qa_samples),
    NUM_MULTIPLE_CHOICE_ANSWERS,
)

if all_scores.shape != expected_score_shape:
    raise ValueError(
        f"Expected score matrix shape {expected_score_shape}, "
        f"found {all_scores.shape}."
    )

if not np.isfinite(all_scores).all():
    raise ValueError(
        "The prediction score matrix contains non-finite values."
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer choices, "
        f"found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Build scored candidate and QA-level prediction records
# ------------------------------------------------------------

scored_candidate_records = []
prediction_rows = []

for sample_index, sample in enumerate(validation_qa_samples):

    scores = all_scores[sample_index]

    predicted_choice = int(
        np.argmax(scores)
    )

    ground_truth_choice = int(
        sample["label"]
    )

    first_row = sample["metadata"]

    if predicted_choice not in range(
        NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"Invalid predicted choice for "
            f"{sample['qa_record_id']}: "
            f"{predicted_choice}"
        )

    if ground_truth_choice not in range(
        NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"Invalid ground-truth choice for "
            f"{sample['qa_record_id']}: "
            f"{ground_truth_choice}"
        )

    candidate_metadata = sample[
        "candidate_metadata"
    ]

    if len(candidate_metadata) != (
        NUM_MULTIPLE_CHOICE_ANSWERS
    ):
        raise ValueError(
            f"QA record {sample['qa_record_id']} has "
            f"{len(candidate_metadata)} candidate metadata rows; "
            f"expected {NUM_MULTIPLE_CHOICE_ANSWERS}."
        )

    # --------------------------------------------------------
    # Candidate-level scored records
    # --------------------------------------------------------

    for candidate_row in candidate_metadata:

        choice_index = int(
            candidate_row["choice_index"]
        )

        if choice_index not in range(
            NUM_MULTIPLE_CHOICE_ANSWERS
        ):
            raise ValueError(
                f"Invalid choice index for "
                f"{sample['qa_record_id']}: "
                f"{choice_index}"
            )

        scored_record = dict(
            candidate_row
        )

        scored_record[
            "prediction_score"
        ] = float(scores[choice_index])

        scored_record[
            "prediction_method"
        ] = REPRESENTATION_VIDEOQA_METHOD

        scored_record[
            "text_embedding_dimension"
        ] = text_embedding_dimension

        scored_record[
            "video_embedding_dimension"
        ] = video_embedding_dimension

        scored_record[
            "fusion_embedding_dimension"
        ] = fusion_embedding_dimension

        scored_record[
            "training_epochs"
        ] = training_epochs

        scored_candidate_records.append(
            scored_record
        )

    # --------------------------------------------------------
    # QA-level prediction record
    # --------------------------------------------------------

    prediction_rows.append(
        {
            "qa_record_id": sample["qa_record_id"],
            "annotation_id": first_row["annotation_id"],
            "split": first_row["split"],
            "video": first_row[video_id_column],
            "question": first_row[question_column],
            "ground_truth_choice": ground_truth_choice,
            "predicted_choice": predicted_choice,
            "choice_correct": (
                predicted_choice
                == ground_truth_choice
            ),
            "ground_truth": first_row[
                f"a{ground_truth_choice}"
            ],
            "prediction": first_row[
                f"a{predicted_choice}"
            ],
            "prediction_score": float(
                scores[predicted_choice]
            ),
            "prediction_method":
                REPRESENTATION_VIDEOQA_METHOD,
            "video_representation_source":
                VIDEO_REPRESENTATION_SOURCE,
            "text_representation_source":
                TEXT_REPRESENTATION_SOURCE,
            "text_embedding_dimension":
                text_embedding_dimension,
            "video_embedding_dimension":
                video_embedding_dimension,
            "fusion_embedding_dimension":
                fusion_embedding_dimension,
            "training_epochs":
                training_epochs,
            "final_training_loss":
                final_training_loss,
        }
    )

scored_candidate_df = pd.DataFrame(
    scored_candidate_records
)

representation_prediction_df = pd.DataFrame(
    prediction_rows
)

# ------------------------------------------------------------
# Validate required prediction columns
# ------------------------------------------------------------

required_prediction_columns = [
    "qa_record_id",
    "annotation_id",
    "split",
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "prediction_score",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
]

required_non_null_prediction_columns = [
    column
    for column in required_prediction_columns
    if column != "final_training_loss"
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in representation_prediction_df.columns
]

if missing_prediction_columns:
    raise ValueError(
        "representation_prediction_df is missing required columns: "
        + ", ".join(missing_prediction_columns)
    )

# ------------------------------------------------------------
# Compute validation checks
# ------------------------------------------------------------

expected_prediction_records = len(
    validation_qa_input_df
)

expected_candidate_rows = (
    expected_prediction_records
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

validation_checks = {
    "prediction_records":
        len(representation_prediction_df),

    "expected_prediction_records":
        expected_prediction_records,

    "candidate_rows":
        len(scored_candidate_df),

    "expected_candidate_rows":
        expected_candidate_rows,

    "unique_qa_records":
        representation_prediction_df[
            "qa_record_id"
        ].nunique(),

    "unique_videos":
        representation_prediction_df[
            "video"
        ].nunique(),

    "missing_values":
        int(
            representation_prediction_df[
                required_non_null_prediction_columns
            ]
            .isna()
            .sum()
            .sum()
        ),

    "duplicate_qa_record_ids":
        int(
            representation_prediction_df[
                "qa_record_id"
            ]
            .duplicated()
            .sum()
        ),

    "invalid_ground_truth_choices":
        int(
            (
                ~representation_prediction_df[
                    "ground_truth_choice"
                ].isin(
                    range(
                        NUM_MULTIPLE_CHOICE_ANSWERS
                    )
                )
            ).sum()
        ),

    "invalid_predicted_choices":
        int(
            (
                ~representation_prediction_df[
                    "predicted_choice"
                ].isin(
                    range(
                        NUM_MULTIPLE_CHOICE_ANSWERS
                    )
                )
            ).sum()
        ),

    "non_boolean_choice_correct":
        int(
            (
                ~representation_prediction_df[
                    "choice_correct"
                ].map(
                    lambda value: isinstance(
                        value,
                        (bool, np.bool_),
                    )
                )
            ).sum()
        ),

    "invalid_prediction_method":
        int(
            (
                representation_prediction_df[
                    "prediction_method"
                ]
                != REPRESENTATION_VIDEOQA_METHOD
            ).sum()
        ),

    "invalid_video_representation_source":
        int(
            (
                representation_prediction_df[
                    "video_representation_source"
                ]
                != VIDEO_REPRESENTATION_SOURCE
            ).sum()
        ),

    "invalid_text_representation_source":
        int(
            (
                representation_prediction_df[
                    "text_representation_source"
                ]
                != TEXT_REPRESENTATION_SOURCE
            ).sum()
        ),

    "invalid_prediction_split":
        int(
            (
                representation_prediction_df[
                    "split"
                ].astype(str)
                != str(evaluation_split)
            ).sum()
        ),

    "non_finite_prediction_scores":
        int(
            (
                ~np.isfinite(
                    representation_prediction_df[
                        "prediction_score"
                    ].astype(float)
                )
            ).sum()
        ),

    "non_finite_candidate_scores":
        int(
            (
                ~np.isfinite(
                    scored_candidate_df[
                        "prediction_score"
                    ].astype(float)
                )
            ).sum()
        ),
}

validation_summary_df = pd.DataFrame(
    validation_checks.items(),
    columns=[
        "Validation Check",
        "Value",
    ],
)

# ------------------------------------------------------------
# Fail on validation errors
# ------------------------------------------------------------

if (
    validation_checks["prediction_records"]
    != validation_checks[
        "expected_prediction_records"
    ]
):
    raise ValueError(
        "Prediction record count does not match "
        "the expected validation QA input count."
    )

if (
    validation_checks["candidate_rows"]
    != validation_checks[
        "expected_candidate_rows"
    ]
):
    raise ValueError(
        "Candidate row count does not match "
        "the expected validation QA-choice count."
    )

if (
    validation_checks["unique_qa_records"]
    != validation_checks[
        "prediction_records"
    ]
):
    raise ValueError(
        "Prediction dataset does not contain "
        "exactly one row per QA record."
    )

zero_required_checks = [
    "missing_values",
    "duplicate_qa_record_ids",
    "invalid_ground_truth_choices",
    "invalid_predicted_choices",
    "non_boolean_choice_correct",
    "invalid_prediction_method",
    "invalid_video_representation_source",
    "invalid_text_representation_source",
    "invalid_prediction_split",
    "non_finite_prediction_scores",
    "non_finite_candidate_scores",
]

failed_checks = {
    check_name: validation_checks[
        check_name
    ]
    for check_name in zero_required_checks
    if validation_checks[check_name] > 0
}

if failed_checks:
    raise ValueError(
        "Prediction dataset validation failed: "
        + ", ".join(
            f"{name}={value}"
            for name, value
            in failed_checks.items()
        )
    )

# ------------------------------------------------------------
# Compute prediction accuracy
# ------------------------------------------------------------

representation_choice_accuracy = float(
    representation_prediction_df[
        "choice_correct"
    ].mean()
)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print(
    "Representation-based prediction datasets "
    "built and validated successfully."
)

print(
    f"Prediction method       : "
    f"{REPRESENTATION_VIDEOQA_METHOD}"
)

print(
    f"Prediction records      : "
    f"{len(representation_prediction_df):,}"
)

print(
    f"Candidate rows scored   : "
    f"{len(scored_candidate_df):,}"
)

print(
    f"Correct predictions     : "
    f"{representation_prediction_df['choice_correct'].sum():,}"
)

print(
    f"Validation accuracy     : "
    f"{representation_choice_accuracy:.3f}"
)

display(validation_summary_df)

print("\nValidation Prediction Preview:")

display(
    representation_prediction_df[
        [
            "qa_record_id",
            "video",
            "question",
            "ground_truth_choice",
            "predicted_choice",
            "choice_correct",
            "ground_truth",
            "prediction",
            "prediction_score",
        ]
    ].head(10)
)



### 🔷 Step 15 — Save Representation-Based VideoQA Results

* Create the configured output directory for the current representation-based VideoQA experiment.
* Save the prediction dataset and validation summary to CSV files.
* Verify that both output files were created successfully.
* Reload the saved files to confirm successful persistence and record integrity.
* Display the output locations and summary of the saved prediction and validation artifacts.

In [ ]:
# ============================================================
# Step 15: Save Representation-Based VideoQA Results
# ============================================================

import shutil
from pathlib import Path

import numpy as np
import pandas as pd

print("Saving representation-based VideoQA results...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "representation_prediction_df",
    "validation_summary_df",
    "representation_choice_accuracy",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
    "fusion_training_history_df",
    "REPRESENTATION_VIDEOQA_METHOD",
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_DRIVE_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_DRIVE_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_DRIVE_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for saving results: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Define supported prediction-method categories
# ------------------------------------------------------------

non_training_prediction_methods = {
    "cosine_similarity",
}

learned_prediction_methods = {
    "fusion_mlp_classifier",
    "interaction_fusion_classifier",
    "gated_fusion_classifier",
    "bilinear_fusion_classifier",
}

supported_prediction_methods = (
    non_training_prediction_methods
    | learned_prediction_methods
)

if REPRESENTATION_VIDEOQA_METHOD not in supported_prediction_methods:
    raise ValueError(
        "Unsupported prediction method: "
        f"{REPRESENTATION_VIDEOQA_METHOD}"
    )

# ------------------------------------------------------------
# Validate result datasets
# ------------------------------------------------------------

if representation_prediction_df.empty:
    raise ValueError(
        "representation_prediction_df is empty."
    )

if validation_summary_df.empty:
    raise ValueError(
        "validation_summary_df is empty."
    )

if fusion_training_history_df is None:
    raise ValueError(
        "fusion_training_history_df is None."
    )

if not isinstance(
    fusion_training_history_df,
    pd.DataFrame,
):
    raise TypeError(
        "fusion_training_history_df must be a pandas DataFrame."
    )

if fusion_training_history_df.empty:
    raise ValueError(
        "fusion_training_history_df is empty."
    )

required_training_history_columns = [
    "epoch",
    "loss",
]

missing_training_history_columns = [
    column
    for column in required_training_history_columns
    if column not in fusion_training_history_df.columns
]

if missing_training_history_columns:
    raise ValueError(
        "fusion_training_history_df is missing required columns: "
        + ", ".join(missing_training_history_columns)
    )

if not np.isfinite(
    float(representation_choice_accuracy)
):
    raise ValueError(
        "representation_choice_accuracy must be finite."
    )

if not (
    0.0
    <= float(representation_choice_accuracy)
    <= 1.0
):
    raise ValueError(
        "representation_choice_accuracy must be between 0 and 1."
    )

# ------------------------------------------------------------
# Validate method-specific training metadata
# ------------------------------------------------------------

if (
    REPRESENTATION_VIDEOQA_METHOD
    in learned_prediction_methods
):

    if int(training_epochs) <= 0:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} requires "
            "training_epochs greater than zero."
        )

    if pd.isna(final_training_loss):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} requires "
            "a valid final_training_loss."
        )

    if not np.isfinite(
        float(final_training_loss)
    ):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} produced "
            "a non-finite final_training_loss."
        )

    if len(fusion_training_history_df) != int(
        training_epochs
    ):
        raise ValueError(
            "Training-history row count does not match "
            f"training_epochs. History rows="
            f"{len(fusion_training_history_df)}, "
            f"training_epochs={training_epochs}."
        )

    learned_losses = (
        fusion_training_history_df["loss"]
        .astype(float)
        .to_numpy()
    )

    if not np.isfinite(learned_losses).all():
        raise ValueError(
            "Learned fusion training history contains "
            "non-finite loss values."
        )

elif (
    REPRESENTATION_VIDEOQA_METHOD
    in non_training_prediction_methods
):

    if int(training_epochs) != 0:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should have "
            "training_epochs equal to zero."
        )

    if not pd.isna(final_training_loss):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should not "
            "have a final training loss."
        )

    if len(fusion_training_history_df) != 1:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should have "
            "one placeholder training-history row."
        )

    placeholder_epoch = int(
        fusion_training_history_df[
            "epoch"
        ].iloc[0]
    )

    placeholder_loss = (
        fusion_training_history_df[
            "loss"
        ].iloc[0]
    )

    if placeholder_epoch != 0:
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should use "
            "placeholder epoch 0."
        )

    if not pd.isna(placeholder_loss):
        raise ValueError(
            f"{REPRESENTATION_VIDEOQA_METHOD} should use "
            "a missing placeholder loss."
        )

# ------------------------------------------------------------
# Validate prediction metadata consistency
# ------------------------------------------------------------

prediction_methods = set(
    representation_prediction_df[
        "prediction_method"
    ]
    .astype(str)
    .unique()
)

if prediction_methods != {
    REPRESENTATION_VIDEOQA_METHOD
}:
    raise ValueError(
        "Prediction dataset contains unexpected methods: "
        f"{sorted(prediction_methods)}"
    )

video_sources = set(
    representation_prediction_df[
        "video_representation_source"
    ]
    .astype(str)
    .unique()
)

if video_sources != {
    VIDEO_REPRESENTATION_SOURCE
}:
    raise ValueError(
        "Prediction dataset contains unexpected video "
        f"representation sources: {sorted(video_sources)}"
    )

text_sources = set(
    representation_prediction_df[
        "text_representation_source"
    ]
    .astype(str)
    .unique()
)

if text_sources != {
    TEXT_REPRESENTATION_SOURCE
}:
    raise ValueError(
        "Prediction dataset contains unexpected text "
        f"representation sources: {sorted(text_sources)}"
    )

# ------------------------------------------------------------
# Build summary dataset
# ------------------------------------------------------------

representation_summary_df = pd.DataFrame(
    [
        {
            "pipeline": "representation_videoqa",
            "experiment_name": EXPERIMENT_NAME,
            "experiment_type": EXPERIMENT_TYPE,
            "prediction_method":
                REPRESENTATION_VIDEOQA_METHOD,
            "video_representation_source":
                VIDEO_REPRESENTATION_SOURCE,
            "text_representation_source":
                TEXT_REPRESENTATION_SOURCE,
            "evaluation_samples":
                len(representation_prediction_df),
            "unique_questions":
                representation_prediction_df[
                    "qa_record_id"
                ].nunique(),
            "unique_videos":
                representation_prediction_df[
                    "video"
                ].nunique(),
            "valid_choice_predictions":
                len(representation_prediction_df),
            "correct_choice_predictions":
                int(
                    representation_prediction_df[
                        "choice_correct"
                    ].sum()
                ),
            "choice_accuracy":
                float(
                    representation_choice_accuracy
                ),
            "text_embedding_dimension":
                int(text_embedding_dimension),
            "video_embedding_dimension":
                int(video_embedding_dimension),
            "fusion_embedding_dimension":
                int(fusion_embedding_dimension),
            "training_epochs":
                int(training_epochs),
            "final_training_loss":
                (
                    None
                    if pd.isna(final_training_loss)
                    else float(final_training_loss)
                ),
        }
    ]
)

# ------------------------------------------------------------
# Validate summary dataset
# ------------------------------------------------------------

required_summary_columns = [
    "pipeline",
    "experiment_name",
    "experiment_type",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "evaluation_samples",
    "unique_questions",
    "unique_videos",
    "valid_choice_predictions",
    "correct_choice_predictions",
    "choice_accuracy",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
]

missing_summary_columns = [
    column
    for column in required_summary_columns
    if column not in representation_summary_df.columns
]

if missing_summary_columns:
    raise ValueError(
        "representation_summary_df is missing required columns: "
        + ", ".join(missing_summary_columns)
    )

if len(representation_summary_df) != 1:
    raise ValueError(
        "representation_summary_df must contain exactly "
        "one summary row."
    )

# ------------------------------------------------------------
# Normalize output paths
# ------------------------------------------------------------

local_output_directory = Path(
    REPRESENTATION_VIDEOQA_LOCAL_DIR
)

drive_output_directory = Path(
    REPRESENTATION_VIDEOQA_DRIVE_DIR
)

local_prediction_path = Path(
    REPRESENTATION_VIDEOQA_PREDICTIONS_CSV
)

local_validation_path = Path(
    REPRESENTATION_VIDEOQA_VALIDATION_CSV
)

local_summary_path = Path(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV
)

drive_prediction_path = Path(
    REPRESENTATION_VIDEOQA_PREDICTIONS_DRIVE_CSV
)

drive_validation_path = Path(
    REPRESENTATION_VIDEOQA_VALIDATION_DRIVE_CSV
)

drive_summary_path = Path(
    REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV
)

# ------------------------------------------------------------
# Create local output directory
# ------------------------------------------------------------

local_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

if not local_output_directory.exists():
    raise FileNotFoundError(
        "Output directory was not created: "
        f"{local_output_directory}"
    )

# ------------------------------------------------------------
# Save local artifacts
# ------------------------------------------------------------

representation_prediction_df.to_csv(
    local_prediction_path,
    index=False,
)

validation_summary_df.to_csv(
    local_validation_path,
    index=False,
)

representation_summary_df.to_csv(
    local_summary_path,
    index=False,
)

local_artifact_paths = [
    local_prediction_path,
    local_validation_path,
    local_summary_path,
]

for local_path in local_artifact_paths:

    if not local_path.exists():
        raise FileNotFoundError(
            f"Failed to create local artifact: {local_path}"
        )

    if local_path.stat().st_size <= 0:
        raise ValueError(
            f"Local artifact is empty: {local_path}"
        )

# ------------------------------------------------------------
# Verify saved local files
# ------------------------------------------------------------

saved_predictions_check_df = pd.read_csv(
    local_prediction_path
)

if len(saved_predictions_check_df) != len(
    representation_prediction_df
):
    raise ValueError(
        "Saved prediction file row count does not match "
        "the in-memory prediction dataset."
    )

saved_validation_check_df = pd.read_csv(
    local_validation_path
)

if len(saved_validation_check_df) != len(
    validation_summary_df
):
    raise ValueError(
        "Saved validation file row count does not match "
        "the in-memory validation summary."
    )

saved_summary_check_df = pd.read_csv(
    local_summary_path
)

if len(saved_summary_check_df) != len(
    representation_summary_df
):
    raise ValueError(
        "Saved summary file row count does not match "
        "the in-memory summary dataset."
    )

missing_saved_summary_columns = [
    column
    for column in required_summary_columns
    if column not in saved_summary_check_df.columns
]

if missing_saved_summary_columns:
    raise ValueError(
        "Saved summary file is missing required columns: "
        + ", ".join(missing_saved_summary_columns)
    )

saved_method = str(
    saved_summary_check_df[
        "prediction_method"
    ].iloc[0]
)

if saved_method != REPRESENTATION_VIDEOQA_METHOD:
    raise ValueError(
        "Saved summary prediction method does not match "
        "the selected method."
    )

# ------------------------------------------------------------
# Promote artifacts to Google Drive
# ------------------------------------------------------------

drive_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

if not drive_output_directory.exists():
    raise FileNotFoundError(
        "Google Drive output directory was not created: "
        f"{drive_output_directory}"
    )

artifact_pairs = [
    (
        local_prediction_path,
        drive_prediction_path,
    ),
    (
        local_validation_path,
        drive_validation_path,
    ),
    (
        local_summary_path,
        drive_summary_path,
    ),
]

for local_path, drive_path in artifact_pairs:

    drive_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        local_path,
        drive_path,
    )

    if not drive_path.exists():
        raise FileNotFoundError(
            f"Failed to promote artifact: {drive_path}"
        )

    if drive_path.stat().st_size != local_path.stat().st_size:
        raise ValueError(
            "Promoted artifact size does not match the "
            f"local artifact: {drive_path}"
        )

print("Artifacts promoted to Google Drive.")

# ------------------------------------------------------------
# Display save summary
# ------------------------------------------------------------

print(
    "Representation-based VideoQA results saved successfully."
)

print(
    f"Local output directory : "
    f"{local_output_directory}"
)

print(
    f"Drive output directory : "
    f"{drive_output_directory}"
)

print(
    f"Prediction method      : "
    f"{REPRESENTATION_VIDEOQA_METHOD}"
)

print(
    f"Prediction file        : "
    f"{local_prediction_path}"
)

print(
    f"Validation file        : "
    f"{local_validation_path}"
)

print(
    f"Summary file           : "
    f"{local_summary_path}"
)

print(
    f"Prediction records     : "
    f"{len(representation_prediction_df):,}"
)

print(
    f"Validation checks      : "
    f"{len(validation_summary_df):,}"
)

print(
    f"Summary records        : "
    f"{len(representation_summary_df):,}"
)



### 🔷 Step 16 — Generate Prediction Summary Report

* Compute summary statistics describing the completed representation-based VideoQA experiment.
* Record the experiment configuration, representation sources, Fusion MLP classifier, and embedding dimensions.
* Summarize prediction counts, correct predictions, incorrect predictions, and multiple-choice accuracy.
* Save the experiment summary as a reusable artifact for downstream evaluation and comparison.
* Verify successful creation of the experiment summary report.


In [ ]:
# ============================================================
# Step 16: Generate Prediction Summary Report
# ============================================================

import pandas as pd
import shutil
from pathlib import Path

print("Generating representation-based VideoQA summary report...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step11_objects = [
    "representation_prediction_df",
    "scored_candidate_df",
    "validation_summary_df",
    "RUN_FULL_EVALUATION_SPLIT",
    "evaluation_split",
    "answer_mode",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_DRIVE_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV",
]

missing_step11_objects = [
    name for name in required_step11_objects
    if name not in globals()
]

if missing_step11_objects:
    raise NameError(
        "Missing required Step 11 objects: "
        + ", ".join(missing_step11_objects)
    )

# ------------------------------------------------------------
# Compute summary metrics
# ------------------------------------------------------------

correct_predictions = int(
    representation_prediction_df["choice_correct"].sum()
)

prediction_records = len(representation_prediction_df)

choice_accuracy = (
    correct_predictions / prediction_records
    if prediction_records > 0
    else 0.0
)

dataset_mode_label = (
    "full_split"
    if RUN_FULL_EVALUATION_SPLIT
    else "development"
)

text_embedding_dimension = int(
    representation_prediction_df["text_embedding_dimension"].iloc[0]
)

video_embedding_dimension = int(
    representation_prediction_df["video_embedding_dimension"].iloc[0]
)

fusion_embedding_dimension = int(
    representation_prediction_df["fusion_embedding_dimension"].iloc[0]
)

training_epochs = (
    int(representation_prediction_df["training_epochs"].iloc[0])
    if "training_epochs" in representation_prediction_df.columns
    else None
)

final_training_loss = (
    representation_prediction_df["final_training_loss"].iloc[0]
    if "final_training_loss" in representation_prediction_df.columns
    else None
)

summary_rows = [
    {"metric": "dataset_mode", "value": dataset_mode_label},
    {"metric": "evaluation_split", "value": evaluation_split},
    {"metric": "answer_mode", "value": answer_mode},
    {"metric": "prediction_method", "value": REPRESENTATION_VIDEOQA_METHOD},
    {"metric": "video_representation_source", "value": VIDEO_REPRESENTATION_SOURCE},
    {"metric": "text_representation_source", "value": TEXT_REPRESENTATION_SOURCE},
    {"metric": "qa_records", "value": prediction_records},
    {"metric": "candidate_rows", "value": len(scored_candidate_df)},
    {"metric": "choice_count", "value": NUM_MULTIPLE_CHOICE_ANSWERS},
    {"metric": "correct_predictions", "value": correct_predictions},
    {"metric": "choice_accuracy", "value": choice_accuracy},
    {"metric": "text_embedding_dimension", "value": text_embedding_dimension},
    {"metric": "video_embedding_dimension", "value": video_embedding_dimension},
    {"metric": "fusion_embedding_dimension", "value": fusion_embedding_dimension},
    {"metric": "training_epochs", "value": training_epochs},
    {"metric": "final_training_loss", "value": final_training_loss},
    {
        "metric": "unique_videos",
        "value": representation_prediction_df["video"].nunique(),
    },
    {
        "metric": "prediction_file",
        "value": str(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    },
    {
        "metric": "validation_file",
        "value": str(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    },
    {
        "metric": "summary_file",
        "value": str(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
    },
]

representation_videoqa_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report locally
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

representation_videoqa_summary_df.to_csv(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
    index=False,
)

if not Path(REPRESENTATION_VIDEOQA_SUMMARY_CSV).exists():
    raise FileNotFoundError(
        f"Failed to create summary file: {REPRESENTATION_VIDEOQA_SUMMARY_CSV}"
    )

# ------------------------------------------------------------
# Verify saved local summary can be reloaded
# ------------------------------------------------------------

saved_summary_check_df = pd.read_csv(REPRESENTATION_VIDEOQA_SUMMARY_CSV)

if len(saved_summary_check_df) != len(representation_videoqa_summary_df):
    raise ValueError(
        "Saved summary file row count does not match "
        "the in-memory summary report."
    )

# ------------------------------------------------------------
# Promote summary report to Google Drive
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
    REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV,
)

if not Path(REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV).exists():
    raise FileNotFoundError(
        f"Failed to promote summary file: "
        f"{REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV}"
    )

# ------------------------------------------------------------
# Display summary report
# ------------------------------------------------------------

print("Representation-based VideoQA summary report saved.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Prediction method  : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Choice accuracy    : {choice_accuracy:.3f}")
print(f"Training epochs    : {training_epochs}")
print(f"Local summary file : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")
print(f"Drive summary file : {REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV}")

display(representation_videoqa_summary_df)



### 🔷 Step 17 — Display Sample Predictions

* Randomly select representative prediction records from the representation-based VideoQA results.
* Display evaluation questions, ground-truth answers, predicted answers, and prediction scores.
* Show the prediction method and representation sources used to generate each prediction.
* Support qualitative assessment of representation-based multiple-choice prediction performance.
* Provide representative examples for experiment verification and debugging.

In [ ]:
# ============================================================
# Step 17: Display Sample Predictions
# ============================================================

import pandas as pd

print("Displaying sample representation-based VideoQA predictions...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step12_objects = [
    "representation_prediction_df",
    "random_seed",
    "dataset_mode_label",
    "evaluation_split",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
]

missing_step12_objects = [
    name for name in required_step12_objects
    if name not in globals()
]

if missing_step12_objects:
    raise NameError(
        "Missing required Step 12 objects: "
        + ", ".join(missing_step12_objects)
    )

if representation_prediction_df.empty:
    raise RuntimeError(
        "representation_prediction_df is empty."
    )

# ------------------------------------------------------------
# Select sample predictions
# ------------------------------------------------------------

sample_count = min(
    10,
    len(representation_prediction_df),
)

sample_prediction_df = (
    representation_prediction_df
    .sample(
        n=sample_count,
        random_state=random_seed,
    )
    .sort_values(
        [
            "choice_correct",
            "video",
            "question",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate display columns
# ------------------------------------------------------------

display_columns = [
    "qa_record_id",
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "prediction_score",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
]

missing_display_columns = [
    col for col in display_columns
    if col not in sample_prediction_df.columns
]

if missing_display_columns:
    raise ValueError(
        "Sample prediction display is missing required columns: "
        + ", ".join(missing_display_columns)
    )

# ------------------------------------------------------------
# Display sample predictions
# ------------------------------------------------------------

print(f"Displaying {sample_count} sample prediction records...")
print(f"Dataset mode                 : {dataset_mode_label}")
print(f"Evaluation split             : {evaluation_split}")
print(f"Prediction method            : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Video representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text representation source   : {TEXT_REPRESENTATION_SOURCE}")
print(f"Text embedding dimension     : {sample_prediction_df['text_embedding_dimension'].iloc[0]}")
print(f"Video embedding dimension    : {sample_prediction_df['video_embedding_dimension'].iloc[0]}")
print(f"Fusion embedding dimension   : {sample_prediction_df['fusion_embedding_dimension'].iloc[0]}")
print(
    f"Sample accuracy              : "
    f"{sample_prediction_df['choice_correct'].mean():.3f}"
)

display(
    sample_prediction_df[
        display_columns
    ]
)



### 🔷 Step 18 — Notebook Summary

* Summarize the completed representation-based VideoQA experiment.
* Report the experiment configuration, representation sources, Fusion MLP classifier, evaluation dataset size, and prediction accuracy.
* Display the locations of the generated prediction, validation, and summary artifacts.
* Confirm that all required experiment artifacts are ready for downstream evaluation in Notebook 08.
* Verify successful completion of the representation-based VideoQA pipeline.




In [ ]:
# ============================================================
# Step 18: Notebook Summary
# ============================================================

print("Notebook 07 complete.")
print("=" * 60)

print("\nRepresentation-Based VideoQA — Configuration")
print("-" * 60)
print(f"Dataset mode                 : {dataset_mode_label}")
print(f"Training split               : train")
print(f"Validation split             : {evaluation_split}")
print(f"Answer mode                  : {answer_mode}")
print(f"Prediction method            : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Video representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text representation source   : {TEXT_REPRESENTATION_SOURCE}")

print("\nInput Dataset Summary")
print("-" * 60)
print(f"Training QA records          : {len(train_qa_input_df):,}")
print(f"Validation QA records        : {len(validation_qa_input_df):,}")
print(f"Validation candidate rows    : {len(scored_candidate_df):,}")
print(f"Training unique videos       : {train_qa_input_df[video_id_column].nunique():,}")
print(f"Validation unique videos     : {representation_prediction_df['video'].nunique():,}")
print(f"Answer choices per question  : {NUM_MULTIPLE_CHOICE_ANSWERS}")

print("\nRepresentation Summary")
print("-" * 60)
print(f"Text embedding dimensions    : {text_embedding_dimension:,}")
print(f"Video embedding dimensions   : {video_embedding_dimension:,}")
print(f"Fusion embedding dimensions  : {fusion_embedding_dimension:,}")
print(f"Training text records        : {len(train_clip_text_df):,}")
print(f"Validation text records      : {len(validation_clip_text_df):,}")
print(f"Training video records       : {len(train_video_representation_df):,}")
print(f"Validation video records     : {len(validation_video_representation_df):,}")

# ------------------------------------------------------------
# Method-specific summary
# ------------------------------------------------------------

if REPRESENTATION_VIDEOQA_METHOD == "cosine_similarity":

    print("\nCosine Similarity Summary")
    print("-" * 60)
    print("Training epochs              : 0")
    print("Final training loss          : not applicable")
    print(
        "Scoring method               : "
        "cosine(video_embedding, answer_choice_embedding)"
    )
    print("Inference split              : validation only")

elif REPRESENTATION_VIDEOQA_METHOD in {
    "fusion_mlp_classifier",
    "interaction_fusion_classifier",
    "gated_fusion_classifier",
    "bilinear_fusion_classifier",
}:

    method_label = REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]

    print(f"\n{method_label} Summary")
    print("-" * 60)
    print(f"Training epochs              : {training_epochs}")
    print(f"Final training loss          : {final_training_loss:.4f}")
    print("Inference split              : validation only")

else:
    raise ValueError(
        f"Unsupported prediction method: "
        f"{REPRESENTATION_VIDEOQA_METHOD}"
    )

print("\nPrediction Results")
print("-" * 60)
print(f"Validation prediction records : {len(representation_prediction_df):,}")
print(f"Correct validation predictions: {int(representation_prediction_df['choice_correct'].sum()):,}")
print(f"Validation accuracy           : {representation_choice_accuracy:.3f}")

print("\nOutput Artifacts")
print("-" * 60)
print(f"Prediction file              : {REPRESENTATION_VIDEOQA_PREDICTIONS_CSV}")
print(f"Validation file              : {REPRESENTATION_VIDEOQA_VALIDATION_CSV}")
print(f"Summary file                 : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")
print(f"Output directory             : {REPRESENTATION_VIDEOQA_LOCAL_DIR}")

print("\nNotebook 07 generated:")

if REPRESENTATION_VIDEOQA_METHOD == "cosine_similarity":

    print(
        "- CLIP cosine-similarity baseline "
        "with no classifier training"
    )

else:

    method_label = REPRESENTATION_VIDEOQA_METHOD_LABELS[
        REPRESENTATION_VIDEOQA_METHOD
    ]

    print(
        f"- {method_label} trained on the training split"
    )

print("- Validation-only representation-based VideoQA prediction dataset")
print("- Prediction validation report")
print("- Representation-based VideoQA summary report")
print("- Sample validation prediction records")

print("\nNotebook 07 outputs are ready for downstream evaluation.")

